# Write-up
## The problem
**Unplanned breakdowns are expensive.** When a part fails on a production line (a breakdown), the line stops, the crew scrambles
for a spare part, and output is lost. If you could see a breakdown coming a day early, you could plan the repair
instead of reacting to it.

**The task.** Oden's brief: take a year of machine data and find **precursors**, patterns in the data that show up
*before* a failure. Each one has to be written as a simple yes/no rule checked every hour ("alarm" or "no alarm"),
so an operator can understand it and trust it. Keep up to three.

**How a rule is judged.** The brief asks for three numbers:
* **Lift:** how much more often the rule fires before failures than in normal running.
* **Precision:** of the alarms it raises, how many are followed by a real failure.
* **Coverage:** of the failures, how many it warned about in time.

To rank rules I fold precision and coverage into one score, **F1** (0 to 1), which is high only when *both* are
high (<a href="#s5-1">section 5.1</a>).

"In time" means inside a set window of hours before the failure, early enough to act (H and W,
<a href="#s5-2">section 5.2</a>).

**What makes it hard.** Failures are rare, and sensors wobble all the time. A rule that fires constantly catches
every failure, but nobody trusts it. A rule that rarely fires is usually right, but misses most failures. And when
many ideas are tried, some will look good by luck, or by quietly using information from the future. The job is to
find rules that are genuinely useful, and prove it on machines they have never seen. So I split the 100 machines
in half: 50 **training machines** to build the rules on, and 50 **test machines** kept aside, used only to grade
them at the end. Every score on this page comes from the test machines (<a href="#s3-1">section 3</a>).

**Why it matters for Oden.** This is a small version of Oden's own
[Agentic Root Cause Analysis](https://engineering.oden.io/blog/agentic-root-cause-analysis) workflow: understand the
operation, narrow the search, come up with ideas, test them fairly, and turn the winners into alarms
(<a href="#s8">section 8</a>).

<a id="data"></a>
### What the data looks like
**Five files, one clock.** Every record carries a **machine number** and an **hour**, so all five files line up
on one shared timeline: 100 machines, every hour from **Jan 1, 2015 to Jan 1, 2016**. Here is one real machine's year:

<div class="timeline">
<b>Machine 1, all of 2015</b>
<div class="svgbox"><svg viewBox="0 0 760 300" width="100%" role="img" aria-label="One real machine's 2015, five rows: hourly sensor readings, 35 error codes, 33 part replacements, 7 breakdowns, and one line of machine details.">
<rect x="192" y="36" width="561" height="66" rx="6" fill="#f3f5f4"/>
<text x="10" y="67" fill="#1d2327" font-weight="600">Sensor readings</text>
<text x="10" y="83" fill="#5d6870" font-size="12">every hour · 4 sensors</text>
<rect x="192" y="112" width="561" height="30" rx="6" fill="#f3f5f4"/>
<text x="10" y="125" fill="#1d2327" font-weight="600">Error codes</text>
<text x="10" y="141" fill="#5d6870" font-size="12">5 types</text>
<rect x="192" y="152" width="561" height="30" rx="6" fill="#f3f5f4"/>
<text x="10" y="165" fill="#1d2327" font-weight="600">Maintenance</text>
<text x="10" y="181" fill="#5d6870" font-size="12">a part replaced</text>
<rect x="192" y="192" width="561" height="30" rx="6" fill="#f3f5f4"/>
<text x="10" y="205" fill="#1d2327" font-weight="600">Breakdowns</text>
<text x="10" y="221" fill="#5d6870" font-size="12">which part failed</text>
<polyline points="200.0,81.7 201.5,80.6 203.0,77.7 204.5,73.6 206.0,78.9 207.5,78.9 209.0,74.5 210.5,77.3 212.0,81.2 213.5,78.9 215.0,84.0 216.5,81.4 218.0,79.5 219.5,83.2 221.0,78.5 222.5,57.1 224.0,47.7 225.5,78.0 227.0,79.7 228.4,87.4 229.9,84.1 231.4,80.1 232.9,81.7 234.4,86.9 235.9,84.7 237.4,67.3 238.9,48.9 240.4,61.6 241.9,81.6 243.4,83.1 244.9,75.9 246.4,87.5 247.9,79.6 249.4,79.2 250.9,84.1 252.4,83.9 253.9,79.8 255.4,79.0 256.9,84.0 258.4,82.0 259.9,83.5 261.4,77.0 262.9,82.0 264.4,76.2 265.9,77.2 267.4,85.4 268.9,75.5 270.4,79.0 271.9,78.9 273.4,77.7 274.9,82.9 276.4,74.1 277.9,76.3 279.4,78.2 280.9,80.5 282.3,80.0 283.8,82.1 285.3,82.1 286.8,78.9 288.3,76.4 289.8,83.6 291.3,83.3 292.8,60.6 294.3,53.7 295.8,76.3 297.3,87.3 298.8,85.9 300.3,79.8 301.8,80.1 303.3,71.7 304.8,75.7 306.3,84.3 307.8,78.0 309.3,74.9 310.8,75.2 312.3,76.8 313.8,83.4 315.3,79.7 316.8,81.9 318.3,85.3 319.8,77.9 321.3,78.1 322.8,82.2 324.3,72.5 325.8,83.1 327.3,80.6 328.8,79.9 330.3,78.6 331.8,75.2 333.3,81.2 334.8,76.4 336.2,78.8 337.7,80.8 339.2,74.6 340.7,78.0 342.2,82.2 343.7,74.7 345.2,80.8 346.7,57.1 348.2,53.6 349.7,82.6 351.2,81.6 352.7,80.8 354.2,79.7 355.7,84.7 357.2,75.2 358.7,78.6 360.2,80.1 361.7,76.4 363.2,84.3 364.7,81.2 366.2,77.2 367.7,79.2 369.2,76.9 370.7,82.4 372.2,81.3 373.7,76.6 375.2,85.5 376.7,79.1 378.2,74.5 379.7,80.8 381.2,73.2 382.7,80.4 384.2,79.9 385.7,80.0 387.2,76.8 388.7,82.8 390.2,79.0 391.6,75.0 393.1,88.9 394.6,79.1 396.1,78.4 397.6,77.1 399.1,76.9 400.6,82.6 402.1,77.0 403.6,81.5 405.1,76.5 406.6,82.4 408.1,80.3 409.6,82.7 411.1,77.7 412.6,86.7 414.1,86.9 415.6,80.5 417.1,82.4 418.6,85.1 420.1,68.4 421.6,83.6 423.1,80.2 424.6,85.3 426.1,84.6 427.6,81.4 429.1,82.7 430.6,82.2 432.1,78.1 433.6,74.5 435.1,49.3 436.6,54.4 438.1,83.4 439.6,83.5 441.1,81.4 442.6,79.8 444.1,85.8 445.5,75.9 447.0,77.8 448.5,75.4 450.0,76.4 451.5,81.1 453.0,74.4 454.5,78.0 456.0,86.0 457.5,87.4 459.0,83.9 460.5,81.9 462.0,78.5 463.5,79.3 465.0,75.9 466.5,76.6 468.0,80.5 469.5,75.0 471.0,83.6 472.5,85.0 474.0,83.9 475.5,77.9 477.0,83.5 478.5,80.3 480.0,77.3 481.5,86.5 483.0,83.2 484.5,74.8 486.0,87.2 487.5,83.3 489.0,92.0 490.5,80.3 492.0,80.2 493.5,81.7 495.0,85.5 496.5,81.1 498.0,84.2 499.5,77.5 500.9,85.9 502.4,80.7 503.9,81.8 505.4,79.7 506.9,84.4 508.4,79.8 509.9,84.8 511.4,71.3 512.9,70.8 514.4,81.4 515.9,86.0 517.4,84.7 518.9,82.0 520.4,76.6 521.9,71.4 523.4,78.8 524.9,77.0 526.4,79.0 527.9,78.5 529.4,81.5 530.9,78.7 532.4,87.0 533.9,73.7 535.4,83.0 536.9,76.2 538.4,81.5 539.9,69.3 541.4,85.4 542.9,80.2 544.4,77.1 545.9,75.6 547.4,82.0 548.9,79.7 550.4,87.1 551.9,80.7 553.4,74.3 554.8,78.1 556.3,75.4 557.8,70.0 559.3,79.4 560.8,78.0 562.3,80.5 563.8,82.4 565.3,84.7 566.8,88.5 568.3,79.8 569.8,82.6 571.3,85.1 572.8,73.0 574.3,81.7 575.8,78.3 577.3,77.8 578.8,81.2 580.3,74.3 581.8,83.1 583.3,83.8 584.8,77.6 586.3,82.1 587.8,83.4 589.3,84.8 590.8,76.9 592.3,80.4 593.8,82.5 595.3,87.4 596.8,78.2 598.3,90.9 599.8,75.4 601.3,75.6 602.8,92.0 604.3,74.1 605.8,83.6 607.3,67.8 608.8,81.3 610.2,81.2 611.7,86.2 613.2,82.1 614.7,81.7 616.2,77.0 617.7,81.1 619.2,73.5 620.7,80.7 622.2,77.1 623.7,82.8 625.2,82.3 626.7,77.8 628.2,82.8 629.7,76.6 631.2,78.1 632.7,82.5 634.2,78.9 635.7,77.5 637.2,75.3 638.7,78.9 640.2,83.1 641.7,75.0 643.2,73.7 644.7,79.8 646.2,81.9 647.7,71.5 649.2,81.7 650.7,83.5 652.2,79.3 653.7,81.4 655.2,72.5 656.7,82.7 658.2,81.2 659.7,81.0 661.2,85.2 662.7,78.5 664.1,81.6 665.6,83.2 667.1,79.4 668.6,81.7 670.1,77.7 671.6,80.4 673.1,86.5 674.6,78.3 676.1,75.1 677.6,79.7 679.1,77.6 680.6,77.9 682.1,84.6 683.6,74.5 685.1,80.1 686.6,87.5 688.1,75.1 689.6,76.1 691.1,78.9 692.6,75.9 694.1,81.6 695.6,78.2 697.1,72.8 698.6,78.8 700.1,78.5 701.6,81.9 703.1,82.4 704.6,78.1 706.1,75.2 707.6,81.5 709.1,77.3 710.6,78.6 712.1,46.0 713.6,58.0 715.1,79.9 716.6,84.5 718.0,75.2 719.5,75.9 721.0,77.6 722.5,79.2 724.0,80.3 725.5,77.9 727.0,79.8 728.5,72.5 730.0,79.8 731.5,84.3 733.0,89.7 734.5,81.9 736.0,81.9 737.5,74.7 739.0,76.2 740.5,82.3 742.0,87.1 743.5,80.8 745.0,82.5" fill="none" stroke="#1d2327" stroke-width="1.3"/>
<text x="745" y="31" text-anchor="end" fill="#5d6870" font-size="12">voltage, daily average (1 of the 4 sensors)</text>
<circle cx="203.0" cy="127" r="4" fill="#b76e00"/>
<circle cx="203.0" cy="127" r="4" fill="#b76e00"/>
<circle cx="204.5" cy="127" r="4" fill="#b76e00"/>
<circle cx="213.5" cy="127" r="4" fill="#b76e00"/>
<circle cx="231.4" cy="127" r="4" fill="#b76e00"/>
<circle cx="235.9" cy="127" r="4" fill="#b76e00"/>
<circle cx="238.9" cy="127" r="4" fill="#b76e00"/>
<circle cx="291.3" cy="127" r="4" fill="#b76e00"/>
<circle cx="294.3" cy="127" r="4" fill="#b76e00"/>
<circle cx="316.8" cy="127" r="4" fill="#b76e00"/>
<circle cx="325.8" cy="127" r="4" fill="#b76e00"/>
<circle cx="333.3" cy="127" r="4" fill="#b76e00"/>
<circle cx="361.7" cy="127" r="4" fill="#b76e00"/>
<circle cx="361.7" cy="127" r="4" fill="#b76e00"/>
<circle cx="376.7" cy="127" r="4" fill="#b76e00"/>
<circle cx="384.2" cy="127" r="4" fill="#b76e00"/>
<circle cx="396.1" cy="127" r="4" fill="#b76e00"/>
<circle cx="409.6" cy="127" r="4" fill="#b76e00"/>
<circle cx="414.1" cy="127" r="4" fill="#b76e00"/>
<circle cx="415.6" cy="127" r="4" fill="#b76e00"/>
<circle cx="438.1" cy="127" r="4" fill="#b76e00"/>
<circle cx="451.5" cy="127" r="4" fill="#b76e00"/>
<circle cx="459.0" cy="127" r="4" fill="#b76e00"/>
<circle cx="550.4" cy="127" r="4" fill="#b76e00"/>
<circle cx="560.8" cy="127" r="4" fill="#b76e00"/>
<circle cx="563.8" cy="127" r="4" fill="#b76e00"/>
<circle cx="581.8" cy="127" r="4" fill="#b76e00"/>
<circle cx="584.8" cy="127" r="4" fill="#b76e00"/>
<circle cx="608.8" cy="127" r="4" fill="#b76e00"/>
<circle cx="629.7" cy="127" r="4" fill="#b76e00"/>
<circle cx="631.2" cy="127" r="4" fill="#b76e00"/>
<circle cx="631.2" cy="127" r="4" fill="#b76e00"/>
<circle cx="631.2" cy="127" r="4" fill="#b76e00"/>
<circle cx="668.6" cy="127" r="4" fill="#b76e00"/>
<circle cx="721.0" cy="127" r="4" fill="#b76e00"/>
<rect x="202.5" y="163.5" width="7" height="7" fill="#0e6170"/>
<rect x="202.5" y="163.5" width="7" height="7" fill="#0e6170"/>
<rect x="224.9" y="163.5" width="7" height="7" fill="#0e6170"/>
<rect x="224.9" y="163.5" width="7" height="7" fill="#0e6170"/>
<rect x="247.4" y="163.5" width="7" height="7" fill="#0e6170"/>
<rect x="247.4" y="163.5" width="7" height="7" fill="#0e6170"/>
<rect x="269.9" y="163.5" width="7" height="7" fill="#0e6170"/>
<rect x="292.3" y="163.5" width="7" height="7" fill="#0e6170"/>
<rect x="314.8" y="163.5" width="7" height="7" fill="#0e6170"/>
<rect x="337.2" y="163.5" width="7" height="7" fill="#0e6170"/>
<rect x="359.7" y="163.5" width="7" height="7" fill="#0e6170"/>
<rect x="382.2" y="163.5" width="7" height="7" fill="#0e6170"/>
<rect x="382.2" y="163.5" width="7" height="7" fill="#0e6170"/>
<rect x="404.6" y="163.5" width="7" height="7" fill="#0e6170"/>
<rect x="427.1" y="163.5" width="7" height="7" fill="#0e6170"/>
<rect x="449.5" y="163.5" width="7" height="7" fill="#0e6170"/>
<rect x="449.5" y="163.5" width="7" height="7" fill="#0e6170"/>
<rect x="472.0" y="163.5" width="7" height="7" fill="#0e6170"/>
<rect x="494.5" y="163.5" width="7" height="7" fill="#0e6170"/>
<rect x="516.9" y="163.5" width="7" height="7" fill="#0e6170"/>
<rect x="561.8" y="163.5" width="7" height="7" fill="#0e6170"/>
<rect x="561.8" y="163.5" width="7" height="7" fill="#0e6170"/>
<rect x="584.3" y="163.5" width="7" height="7" fill="#0e6170"/>
<rect x="606.7" y="163.5" width="7" height="7" fill="#0e6170"/>
<rect x="606.7" y="163.5" width="7" height="7" fill="#0e6170"/>
<rect x="629.2" y="163.5" width="7" height="7" fill="#0e6170"/>
<rect x="629.2" y="163.5" width="7" height="7" fill="#0e6170"/>
<rect x="651.7" y="163.5" width="7" height="7" fill="#0e6170"/>
<rect x="651.7" y="163.5" width="7" height="7" fill="#0e6170"/>
<rect x="674.1" y="163.5" width="7" height="7" fill="#0e6170"/>
<rect x="696.6" y="163.5" width="7" height="7" fill="#0e6170"/>
<rect x="719.0" y="163.5" width="7" height="7" fill="#0e6170"/>
<rect x="741.5" y="163.5" width="7" height="7" fill="#0e6170"/>
<path d="M200.0 201 l12 12 M212.0 201 l-12 12" stroke="#b03a3a" stroke-width="3"/>
<line x1="206.0" y1="44" x2="206.0" y2="198" stroke="#b03a3a" stroke-width="1" stroke-dasharray="3 3" opacity=".6"/>
<path d="M289.8 201 l12 12 M301.8 201 l-12 12" stroke="#b03a3a" stroke-width="3"/>
<line x1="295.8" y1="44" x2="295.8" y2="198" stroke="#b03a3a" stroke-width="1" stroke-dasharray="3 3" opacity=".6"/>
<path d="M357.2 201 l12 12 M369.2 201 l-12 12" stroke="#b03a3a" stroke-width="3"/>
<line x1="363.2" y1="44" x2="363.2" y2="198" stroke="#b03a3a" stroke-width="1" stroke-dasharray="3 3" opacity=".6"/>
<path d="M447.0 201 l12 12 M459.0 201 l-12 12" stroke="#b03a3a" stroke-width="3"/>
<line x1="453.0" y1="44" x2="453.0" y2="198" stroke="#b03a3a" stroke-width="1" stroke-dasharray="3 3" opacity=".6"/>
<path d="M559.3 201 l12 12 M571.3 201 l-12 12" stroke="#b03a3a" stroke-width="3"/>
<line x1="565.3" y1="44" x2="565.3" y2="198" stroke="#b03a3a" stroke-width="1" stroke-dasharray="3 3" opacity=".6"/>
<path d="M626.7 201 l12 12 M638.7 201 l-12 12" stroke="#b03a3a" stroke-width="3"/>
<line x1="632.7" y1="44" x2="632.7" y2="198" stroke="#b03a3a" stroke-width="1" stroke-dasharray="3 3" opacity=".6"/>
<path d="M716.5 201 l12 12 M728.5 201 l-12 12" stroke="#b03a3a" stroke-width="3"/>
<line x1="722.5" y1="44" x2="722.5" y2="198" stroke="#b03a3a" stroke-width="1" stroke-dasharray="3 3" opacity=".6"/>
<text x="221.0" y="240" text-anchor="middle" fill="#5d6870" font-size="12">Jan</text>
<text x="267.4" y="240" text-anchor="middle" fill="#5d6870" font-size="12">Feb</text>
<text x="309.3" y="240" text-anchor="middle" fill="#5d6870" font-size="12">Mar</text>
<text x="355.7" y="240" text-anchor="middle" fill="#5d6870" font-size="12">Apr</text>
<text x="400.6" y="240" text-anchor="middle" fill="#5d6870" font-size="12">May</text>
<text x="447.0" y="240" text-anchor="middle" fill="#5d6870" font-size="12">Jun</text>
<text x="492.0" y="240" text-anchor="middle" fill="#5d6870" font-size="12">Jul</text>
<text x="538.4" y="240" text-anchor="middle" fill="#5d6870" font-size="12">Aug</text>
<text x="584.8" y="240" text-anchor="middle" fill="#5d6870" font-size="12">Sep</text>
<text x="629.7" y="240" text-anchor="middle" fill="#5d6870" font-size="12">Oct</text>
<text x="676.1" y="240" text-anchor="middle" fill="#5d6870" font-size="12">Nov</text>
<text x="721.0" y="240" text-anchor="middle" fill="#5d6870" font-size="12">Dec</text>
<rect x="10" y="256" width="735" height="34" rx="6" fill="#e3f0f1"/>
<text x="22" y="278" fill="#0e6170"><tspan font-weight="600">Machine details:</tspan> machine 1 · model 3 · 18 years old  <tspan fill="#5d6870">(one line per machine, never changes)</tspan></text>
</svg></div>
</div>

**What to notice:** the voltage jumps two days before machine 1's voltage part (comp1) breaks down in early March.
But it also jumps twice in mid-January with no breakdown after it. Telling those two kinds of jump apart is the
whole problem. (Every ✗ also has a square above it: the repair is logged as a part replaced.)

| File | One row means | Rows | Its job |
|---|---|---|---|
| **Sensor readings** | one machine, one hour: voltage, rotation, pressure, vibration | 876,100 (100 machines × 8,761 hours) | clues |
| **Error codes** | the machine logged an error (types error1–error5) | 3,919 | clues |
| **Maintenance** | a part (comp1–comp4) was replaced | 3,286 (starts mid-2014, so each part's age is known on day one) | how long since each part was replaced |
| **Breakdowns** | a part (comp1–comp4) failed | 761 rows → **719 breakdowns** (two parts failing in the same hour count once) | what we want to warn about |
| **Machine details** | the machine's model (1–4) and age (0–20 years) | 100 | background |

In data science terms, the breakdowns are the **target** (what we predict) and the other files are the
**features** (the clues). Where the data comes from and how to reproduce it: <a href="#s4">section 4</a>.

### What the brief asked for, and where it's answered
| | Brief asks for | In plain words |
|---|---|---|
| 1 | Your shortlist of surviving precursors | [The warning signs that made the cut](#s1) |
| 2 | The candidates you rejected and why | [The ideas I threw out, and why](#s2) · [the ML models](#s2-2) |
| 3 | A brief description of your assumptions | [What I assumed](#s3) · [how the test was kept fair](#s3-1) |
| 4 | The data source, with enough detail to reproduce | [Where the data came from](#s4) |
| 5 | Your main design choices (H, W, thresholds, how you defined normal) | [How rules were scored](#s5-1) · [when an alarm counts](#s5-2) · [what "normal" means](#s5-3) · [what "abnormal" means](#s5-4) |
| 6 | If you run out of time: what you'd do next | [What I'd do next](#s6) |
| 7 | If you used AI tools: transcripts and notes on corrections | [How I used AI, and where I corrected it](#s7) |
| 8 | Background: build a small version of Oden's root-cause workflow | [How this mirrors Oden's own process](#s8) |
| 9 | (closing) | [Impact: what C09 gives you that you didn't have before](#s9) |
| | Appendix | [Key terms](#appx-a) · [how the project is built](#appx-b) · [what each notebook found](#appx-c) |

In [1]:
# --- Setup ----------------------------------------------------------------------------
# Libraries, plus the project's helper files in ../src (data.py, detectors.py, evaluation.py).
# Every analysis function this notebook calls lives there, in numbered sections (§1 ... §9).
import json, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found")  # harmless: kagglehub's progress bar

import pandas as pd
import plotly.io as pio

# Two outputs per chart: Plotly's own MIME type (rendered natively by VS Code and JupyterLab, works
# offline) plus an HTML fallback for classic Jupyter (loads plotly.js from a CDN).
pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.max_colwidth", 120, "display.width", 200)  # show long rule texts in full

ROOT = Path.cwd().parent            # project root (this notebook lives in notebooks/)
sys.path.insert(0, str(ROOT / "src"))  # so `from data import ...` finds ../src/data.py

In [2]:
# --- The saved results of notebooks 01-05 ------------------------------------------------
# This notebook computes nothing: it reads what notebooks 01-05 saved to outputs/. Run them first, in order.
from evaluation import load_results, check_results   # src/evaluation.py §9

N1 = load_results("01_dataset_and_parameters")        # dataset profile, timestamp audit, look-alikes
N2 = load_results("02_candidates")                    # the ten candidates
N3 = load_results("03_ml_benchmark")                  # the ML models' scores
N4 = load_results("04_test_and_select")               # every candidate's scores, gates and verdict; the H x W grid
N5 = load_results("05_audit")                         # the audits
K = N4["kinds"]["any"]                                # "any failure", the main event type

# One short name per rule, used in every table and chart (the saved titles of C08 and C09 are longer).
NAME = {"C08": "Drift + any error", "C09": "Matched signature"}
rule_name = lambda c: NAME.get(c["cid"], c["title"])

In [3]:
# --- Charts -----------------------------------------------------------------------------
# Two kinds, both from saved results only: charts notebooks 01 and 04 already drew (read back from their saved
# outputs, exactly as drawn), and charts drawn here from the five results files. Nothing is recomputed.
import numpy as np
import plotly.graph_objects as go

BLUE, DARK, GREY, ORANGE = "#2a78d6", "#4a5358", "#b4bcc0", "#eb6834"

def notebook_figure(nb, title=None, trace=None):
    """A chart exactly as notebook `nb` drew it, read back from that notebook's saved output.
    Found by its title, or (for an untitled chart) by the name of one of its lines."""
    for c in json.load(open(ROOT / "notebooks" / f"{nb}.ipynb"))["cells"]:
        for o in c.get("outputs", []):
            spec = o.get("data", {}).get("application/vnd.plotly.v1+json")
            if not spec:
                continue
            if (title and title in json.dumps(spec["layout"].get("title", {}), ensure_ascii=False)) or \
               (trace and any(t.get("name") == trace for t in spec["data"])):
                return go.Figure(spec).update_layout(template="plotly_white")
    raise KeyError(f"no chart {title or trace!r} in notebook {nb}: run that notebook first")

def look(fig, title, height=420):
    """The house style for charts drawn here."""
    return fig.update_layout(title=dict(text=title, font=dict(size=15)), template="plotly_white", height=height,
                             margin=dict(l=60, r=30, t=60, b=50), legend=dict(orientation="h", y=-0.18))

<a id="s1"></a>
## 1 · Shortlist: the warning signs that made the cut
*Brief: your shortlist of surviving precursors.*

**The story in plain words.** Each machine has four parts (comp1–comp4). Each part has one sensor that watches it
and one error code it can log. About **two days** before a part breaks, its sensor moves off its normal level and
stays there. About **one day** before, that part's own error code shows up.

The catch: sensors also move off for no reason, about **470 times a year per part**, and then settle back. Think of a
fever: most pass on their own. **The error code is what tells you which fever is serious.**

### Why the error code matters

A *drift* is a sensor running a fever: clearly off that machine's normal, and staying there for at least a day.
The chart counts every drift in 2015. Blue ones were followed by a failure (*real*); grey ones settled back on their
own (*look-alikes*).

In [4]:
# The key property (notebook 01), pooled over the four parts: every drift in a part's own signal, split into real
# drifts (a failure followed) and look-alikes (none did); then only the drifts during which an error code (any code)
# was logged. Each bar adds up to 100% of its drifts. How a drift is defined is written under the chart.
LA = pd.DataFrame(N1["lookalikes"]).set_index("index")
real, fake = LA.episodes_ending_in_failure, LA.episodes_without_failure
rows = {"sensor drift alone": (real.sum(), fake.sum()),
        "drift + an error code": ((real * LA.share_with_error_failure).sum(),
                                              (fake * LA.share_with_error_none).sum())}
fig = go.Figure()
for name, color, k in [("real drift (a failure followed)", BLUE, 0), ("look-alike (no failure followed)", GREY, 1)]:
    vals = [v[k] for v in rows.values()]
    fig.add_bar(y=list(rows), x=vals, name=name, orientation="h", marker_color=color,
                text=[f"{v:,.0f}" for v in vals], textposition="inside", insidetextanchor="middle",
                textfont=dict(color="white" if k == 0 else "#1d2327", size=13), hoverinfo="skip")
for label, (r, f) in rows.items():
    fig.add_annotation(y=label, x=r + f, text=f"<b>{r / (r + f):.0%} real</b>", showarrow=False, xanchor="left",
                       xshift=8, font=dict(size=14, color=BLUE))
kept_real = rows["drift + an error code"][0] / real.sum()
removed = 1 - rows["drift + an error code"][1] / fake.sum()
look(fig, f"Requiring an error code removes {removed:.0%} of look-alikes and keeps {kept_real:.0%} of real drifts", 300)
fig.add_annotation(x=0, y=-0.36, xref="paper", yref="paper", xanchor="left", yanchor="top", align="left",
                   showarrow=False, font=dict(size=11.5, color="#4a5358"), text=(
    "<b>What counts as a drift:</b> a part's own sensor stays clearly off that machine's normal (about 0.7 to 1.2 typical "
    "wobbles, averaged over 12 hours)<br>for at least a full day. <b>Real:</b> that part failed soon after. "
    "<b>With an error code:</b> an error code was logged during the drift."))
fig.update_layout(barmode="stack", bargap=0.35, height=470, margin=dict(t=95, b=140),
                  legend=dict(orientation="h", y=1.02, yanchor="bottom", x=0, traceorder="normal"))
fig.update_xaxes(title="number of drifts in a part's own signal (all machines, 2015)", range=[0, 3000])
fig.update_yaxes(autorange="reversed", title=None)

**What the chart shows: a drift alone can't be trusted, but a drift with an error code usually can.** On the sensor alone, only about 1 in 4 drifts is real. Requiring an error code throws out most look-alikes and
keeps almost every real drift, so about 7 in 10 are real.

**The ten ideas I tested.** Seven are the obvious ones anyone would try first (C01–C07). Three combine clues,
based on what the data showed (C08–C10):

| ID | Idea | What it alarms on | Result |
|---|---|---|---|
| C01 | Voltage creeps up | a machine's voltage rises well above its normal level (an electrical fault) | rejected |
| C02 | Rotation sags | rotation speed drops well below its normal level (friction or wear) | rejected |
| C03 | Pressure builds | pressure rises well above its normal level (a blockage) | rejected |
| C04 | Vibration rises | vibration rises well above its normal level (bearing wear) | rejected |
| C05 | Any signal drifts | any of the four sensors moves well past its normal level | rejected |
| C06 | Any error code | the machine logs any error code | rejected |
| C07 | Overdue maintenance | a part has gone much longer than usual since its last replacement | rejected (failed a check) |
| C08 | Drift + any error | any sensor reads off **and** any error code was logged recently | **kept** |
| C09 | Matched signature | a part's own sensor reads off **and** that same part's own error code was logged recently | **kept, #1** |
| C10 | Drift on an aging part | a part's own sensor reads off **and** the part is overdue for replacement | **kept** |

**Shortlisted down to three.** Every idea had to pass four checks ([section 2](#s2)), then the top three by score
were kept. Here they are, scored on the 50 test machines the rules never saw:

| | Warning sign | In plain words | Breakdowns caught | Alarms that were real | Warns ahead | Role |
|---|---|---|---|---|---|---|
| **1** | **C09 · Matched signature** | a part's sensor reads off **and** that same part logs its own error code | **99%** | **90%** | ~24 h | **the alarm to deploy** |
| 2 | C08 · Drift + any error | any sensor reads off and any error code appears | 91% | 63% | ~24 h | backup, where we don't yet know which code goes with which part |
| 3 | C10 · Drift on an aging part | a part's sensor reads off and the part is overdue for replacement | 89% | 41% | ~43 h | early watchlist: warns sooner, wrong more often |

All three easily pass the brief's lift test: each fires far more often before a failure than in normal running.

**What the three rules look at.** Every rule needs two clues at once. What separates them is whether the two
clues must come from the **same part**. Each part has its own sensor and its own error code (comp1 voltage · error1,
comp2 rotation · error2/3, comp3 pressure · error4, comp4 vibration · error5), so C09 is really four small rules, one
per part, each asking "is *this* part's sensor off *and* did *this* part's code fire?"

<div class="timeline">
<b>Which clues each rule needs (worked example: comp3, the pressure part)</b>
<div class="svgbox">
<svg viewBox="0 0 760 226" width="100%" role="img" aria-label="Three panels, one per rule. Each shows four parts and three clue columns: own sensor, own error code, part age. C09 connects a part's own sensor to that same part's error code. C08 connects any sensor to any error code across parts. C10 connects a part's own sensor to that part's age, skipping the error code.">
<text x="10" y="16" fill="#1d2327" font-weight="600" font-size="13">C09 · Matched signature</text>
<text x="82" y="38" text-anchor="middle" fill="#5d6870" font-size="10">own sensor</text>
<text x="144" y="38" text-anchor="middle" fill="#5d6870" font-size="10">own error code</text>
<text x="206" y="38" text-anchor="middle" fill="#5d6870" font-size="10">part age</text>
<text x="10" y="63" fill="#5d6870" font-size="11">comp1</text>
<rect x="55" y="48" width="54" height="22" rx="5" fill="#f3f5f6" stroke="#dfe3e1" stroke-width="1"/>
<rect x="117" y="48" width="54" height="22" rx="5" fill="#f3f5f6" stroke="#dfe3e1" stroke-width="1"/>
<rect x="179" y="48" width="54" height="22" rx="5" fill="#f3f5f6" stroke="#dfe3e1" stroke-width="1"/>
<text x="10" y="93" fill="#5d6870" font-size="11">comp2</text>
<rect x="55" y="78" width="54" height="22" rx="5" fill="#f3f5f6" stroke="#dfe3e1" stroke-width="1"/>
<rect x="117" y="78" width="54" height="22" rx="5" fill="#f3f5f6" stroke="#dfe3e1" stroke-width="1"/>
<rect x="179" y="78" width="54" height="22" rx="5" fill="#f3f5f6" stroke="#dfe3e1" stroke-width="1"/>
<text x="10" y="123" fill="#5d6870" font-size="11">comp3</text>
<rect x="55" y="108" width="54" height="22" rx="5" fill="#e3f0f1" stroke="#0e6170" stroke-width="2"/>
<rect x="117" y="108" width="54" height="22" rx="5" fill="#e3f0f1" stroke="#0e6170" stroke-width="2"/>
<rect x="179" y="108" width="54" height="22" rx="5" fill="#f3f5f6" stroke="#dfe3e1" stroke-width="1"/>
<text x="10" y="153" fill="#5d6870" font-size="11">comp4</text>
<rect x="55" y="138" width="54" height="22" rx="5" fill="#f3f5f6" stroke="#dfe3e1" stroke-width="1"/>
<rect x="117" y="138" width="54" height="22" rx="5" fill="#f3f5f6" stroke="#dfe3e1" stroke-width="1"/>
<rect x="179" y="138" width="54" height="22" rx="5" fill="#f3f5f6" stroke="#dfe3e1" stroke-width="1"/>
<line x1="109" y1="119" x2="117" y2="119" stroke="#0e6170" stroke-width="2.5"/>
<text x="10" y="186" fill="#1d2327" font-weight="600" font-size="11.5">Both clues from the SAME part</text>
<text x="10" y="201" fill="#5d6870" font-weight="400" font-size="11">comp3's pressure off AND error4 fired</text>
<text x="10" y="216" fill="#5d6870" font-weight="400" font-size="11">9 in 10 alarms real</text>
<text x="265" y="16" fill="#1d2327" font-weight="600" font-size="13">C08 · Drift + any error</text>
<text x="337" y="38" text-anchor="middle" fill="#5d6870" font-size="10">own sensor</text>
<text x="399" y="38" text-anchor="middle" fill="#5d6870" font-size="10">own error code</text>
<text x="461" y="38" text-anchor="middle" fill="#5d6870" font-size="10">part age</text>
<text x="265" y="63" fill="#5d6870" font-size="11">comp1</text>
<rect x="310" y="48" width="54" height="22" rx="5" fill="#f3f5f6" stroke="#dfe3e1" stroke-width="1"/>
<rect x="372" y="48" width="54" height="22" rx="5" fill="#e3f0f1" stroke="#0e6170" stroke-width="2"/>
<rect x="434" y="48" width="54" height="22" rx="5" fill="#f3f5f6" stroke="#dfe3e1" stroke-width="1"/>
<text x="265" y="93" fill="#5d6870" font-size="11">comp2</text>
<rect x="310" y="78" width="54" height="22" rx="5" fill="#f3f5f6" stroke="#dfe3e1" stroke-width="1"/>
<rect x="372" y="78" width="54" height="22" rx="5" fill="#f3f5f6" stroke="#dfe3e1" stroke-width="1"/>
<rect x="434" y="78" width="54" height="22" rx="5" fill="#f3f5f6" stroke="#dfe3e1" stroke-width="1"/>
<text x="265" y="123" fill="#5d6870" font-size="11">comp3</text>
<rect x="310" y="108" width="54" height="22" rx="5" fill="#e3f0f1" stroke="#0e6170" stroke-width="2"/>
<rect x="372" y="108" width="54" height="22" rx="5" fill="#f3f5f6" stroke="#dfe3e1" stroke-width="1"/>
<rect x="434" y="108" width="54" height="22" rx="5" fill="#f3f5f6" stroke="#dfe3e1" stroke-width="1"/>
<text x="265" y="153" fill="#5d6870" font-size="11">comp4</text>
<rect x="310" y="138" width="54" height="22" rx="5" fill="#f3f5f6" stroke="#dfe3e1" stroke-width="1"/>
<rect x="372" y="138" width="54" height="22" rx="5" fill="#f3f5f6" stroke="#dfe3e1" stroke-width="1"/>
<rect x="434" y="138" width="54" height="22" rx="5" fill="#f3f5f6" stroke="#dfe3e1" stroke-width="1"/>
<line x1="364" y1="119" x2="372" y2="59" stroke="#0e6170" stroke-width="2.5" stroke-dasharray="5 4"/>
<text x="265" y="186" fill="#1d2327" font-weight="600" font-size="11.5">Any sensor + any error code</text>
<text x="265" y="201" fill="#5d6870" font-weight="400" font-size="11">the clues needn't match a part</text>
<text x="265" y="216" fill="#5d6870" font-weight="400" font-size="11">6 in 10 alarms real</text>
<text x="520" y="16" fill="#1d2327" font-weight="600" font-size="13">C10 · Drift on an aging part</text>
<text x="592" y="38" text-anchor="middle" fill="#5d6870" font-size="10">own sensor</text>
<text x="654" y="38" text-anchor="middle" fill="#5d6870" font-size="10">own error code</text>
<text x="716" y="38" text-anchor="middle" fill="#5d6870" font-size="10">part age</text>
<text x="520" y="63" fill="#5d6870" font-size="11">comp1</text>
<rect x="565" y="48" width="54" height="22" rx="5" fill="#f3f5f6" stroke="#dfe3e1" stroke-width="1"/>
<rect x="627" y="48" width="54" height="22" rx="5" fill="#f3f5f6" stroke="#dfe3e1" stroke-width="1"/>
<rect x="689" y="48" width="54" height="22" rx="5" fill="#f3f5f6" stroke="#dfe3e1" stroke-width="1"/>
<text x="520" y="93" fill="#5d6870" font-size="11">comp2</text>
<rect x="565" y="78" width="54" height="22" rx="5" fill="#f3f5f6" stroke="#dfe3e1" stroke-width="1"/>
<rect x="627" y="78" width="54" height="22" rx="5" fill="#f3f5f6" stroke="#dfe3e1" stroke-width="1"/>
<rect x="689" y="78" width="54" height="22" rx="5" fill="#f3f5f6" stroke="#dfe3e1" stroke-width="1"/>
<text x="520" y="123" fill="#5d6870" font-size="11">comp3</text>
<rect x="565" y="108" width="54" height="22" rx="5" fill="#e3f0f1" stroke="#0e6170" stroke-width="2"/>
<rect x="627" y="108" width="54" height="22" rx="5" fill="#f3f5f6" stroke="#dfe3e1" stroke-width="1"/>
<rect x="689" y="108" width="54" height="22" rx="5" fill="#e3f0f1" stroke="#0e6170" stroke-width="2"/>
<text x="520" y="153" fill="#5d6870" font-size="11">comp4</text>
<rect x="565" y="138" width="54" height="22" rx="5" fill="#f3f5f6" stroke="#dfe3e1" stroke-width="1"/>
<rect x="627" y="138" width="54" height="22" rx="5" fill="#f3f5f6" stroke="#dfe3e1" stroke-width="1"/>
<rect x="689" y="138" width="54" height="22" rx="5" fill="#f3f5f6" stroke="#dfe3e1" stroke-width="1"/>
<path d="M610 130 C610 138, 698 138, 698 130" fill="none" stroke="#0e6170" stroke-width="2.5"/>
<text x="520" y="186" fill="#1d2327" font-weight="600" font-size="11.5">Own sensor + that part's age</text>
<text x="520" y="201" fill="#5d6870" font-weight="400" font-size="11">no error code needed: earlier (~43 h)</text>
<text x="520" y="216" fill="#5d6870" font-weight="400" font-size="11">but only 4 in 10 alarms real</text>
</svg>
</div>
</div>

<div class="dots-row">
  <div class="dots-card"><b>Out of every 100 breakdowns</b>
    <div class="dots" role="img" aria-label="99 of 100 breakdowns caught, 1 missed"><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i class="miss"></i></div>
    <span>C09 warned about 99. One slipped through (red ring). On the 50 test machines: 346 of 350.</span></div>
  <div class="dots-card"><b>Out of every 10 alarms</b>
    <div class="dots" role="img" aria-label="9 of 10 alarms real, 1 false"><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i></i><i class="false"></i></div>
    <span>9 were followed by a real breakdown, 1 was a false alarm (amber ring). About 1.5 alarms per machine per month.</span></div>
</div>

**Recommendation: deploy C09 alone.** The brief allows up to three, so three are listed, but adding the other two
barely helps. On the 50 test machines:

| Added to C09 | Extra breakdowns caught | Extra false alarms |
|---|---|---|
| C08 (drift + any error) | 1 | 462 |
| C10 (drift on an aging part) | 3 | 1,374 |

C10's one advantage is timing: it warns about 20 hours earlier. It's worth adding only if a day isn't enough to
act, say a spare part that takes two days to arrive.

The exact rule definitions are in [appendix A.2](#appx-a2).

<a id="s2"></a>
## 2 · Rejected candidates, and why
*Brief: the candidates you rejected and why.*

Each of the ten ideas from [section 1](#s1) was tried with several settings (how strict the line is, how far back to look, and so on) and
its best setting kept (appendix C.2). Then each idea had to pass **four checks** before it could compete:

1. **Not luck:** it must clearly beat scrambled copies of itself (same number of alarms, shifted randomly in time).
   Rules built from pure random noise score about 0.08 out of 1; C09 scores 0.94.
2. **It fires more before breakdowns** than in normal running, at least twice as often (lift ≥ 2).
3. **It catches a meaningful share** of breakdowns, at least 5% (coverage ≥ 5%).
4. **No peeking at the future:** set up on January–June only, it must still work on July–December.

Nine of the ten passed, and the top three by score were kept. Why the other seven lost:

* **Watching one sensor at a time (C01–C04):** real signals, but fooled by the false "fevers". Only about 1 in 4
  alarms is real.
* **Watching all four sensors at once (C05):** catches 98% of breakdowns, but nearly 3 in 4 alarms are false. A crew
  would learn to ignore it.
* **Alarming on any error code (C06):** misses about half of all breakdowns.
* **Alarming when a part is overdue for service (C07):** the only idea that failed a check. It looked good only
  because it was quietly using information from the future; with past data alone it caught nothing (appendix C.5).

<a id="s2-2"></a>
### 2.2 · Also rejected: a machine-learning model
**In plain words:** I also trained three machine-learning models, each given all 32 inputs the rules could use.
The best one was right only 46% of the time it raised an alarm. The simple two-clue rule was both more accurate
and easier to explain.

The chart below scores the models exactly like the rules: same warning window, same 50 test machines.

In [5]:
# The three kept rules (notebook 04) next to the three ML models (notebook 03): same window, same holdout machines.
rows = [dict(name=f"rule {c['cid']}", kind="C09" if c["cid"] == "C09" else "rule", **{k: c["hold"][k] for k in
        ("f1", "precision", "coverage")}) for c in K["concepts"] if c["status"] == "kept"]
rows += [dict(name=m["detector"], kind="ML", f1=m["F1"], precision=m["precision"], coverage=m["coverage"])
         for m in N3["models"]]
D = pd.DataFrame(rows).sort_values("f1")
fig = go.Figure(go.Bar(x=D.f1, y=D.name, orientation="h",
                       marker_color=[{"C09": BLUE, "rule": DARK, "ML": ORANGE}[k] for k in D.kind],
                       text=[f"F1 {f:.2f}  ·  precision {p:.0%}  ·  coverage {c:.0%}"
                             for f, p, c in zip(D.f1, D.precision, D.coverage)],
                       textposition="outside", hoverinfo="skip"))
look(fig, "Simple rules against ML models on the test machines (blue/grey = rules, orange = ML)", 360)
fig.update_xaxes(range=[0, 1.45], title="F1", tickvals=[0, 0.25, 0.5, 0.75, 1.0]); fig.update_yaxes(title=None)

**Why a simple rule beats a 32-input model here:**
* **The warning sign is very specific:** *a part's own sensor reads off while its own error code is logged*. C09
  looks for exactly that. A general model has to work out that pairing on its own, and the simplest model can't
  represent it at all.
* **The models catch nearly every failure (97–100%), but raise far more false alarms.** One of them also leans on
  "hours since maintenance", the same misleading clue that sank C07.

**Fair caveat:** I didn't fine-tune the models, so this is a sanity check, not the best ML could do. My rule of
thumb: an alarm people can understand and trust wins unless a model clearly beats it, and none did.

<a id="s3"></a>
## 3 · Assumptions
*Brief: a brief description of your assumptions.*

* **The failure time in the log is accurate.** Failures are logged once a day at 6 a.m., so I checked hour by hour
  that no after-the-fact data sneaks in before that time. None does.
* **The repair logged with a failure is part of the failure,** not a warning sign. Otherwise "a repair was logged"
  would look like a perfect predictor.
* **Rules only use the past.** At any hour, a rule can only look at readings from before that hour.
* **Machines never switch off** (they always run), so there is no "idle" time to set aside.
* **The machines are similar enough** that rules built on half of them carry over to the other half.
* **The false "fevers" are real behaviour** of the machines, perhaps failures that a crew prevented. I didn't
  relabel them.

<a id="s3-1"></a>
### The fair test: training machines and test machines
I built every rule on 50 **training machines** (the *dev* set), then graded it once on 50 **test machines** (the
*holdout* set) that never influenced any choice: not which ideas, not their settings, not the warning window. Every
score in sections 1 and 2 comes from the test machines. A few edge hours aren't graded so nothing scores by accident: the
failure hour itself (it holds the repair record), the day after a repair, and the first and last days of the year.

<a id="s4"></a>
## 4 · Data source and reproduction
*Brief: the data source, with enough detail to reproduce.*

**What it is:** Microsoft's public **Azure Predictive Maintenance** sample: the five files shown in
<a href="#data">what the data looks like</a>, hourly from Jan 1, 2015 to Jan 1, 2016, with no gaps or missing values.

**One honest caveat:** it's simulated data, and unusually tidy. Warning signs land at exactly 48 and 24 hours before
a breakdown. A real factory will be messier, so the exact percentages are optimistic.

**To reproduce:**
* **Data:** [Microsoft Azure Predictive Maintenance on Kaggle](https://www.kaggle.com/datasets/arnabbiswas1/microsoft-azure-predictive-maintenance),
  version 3, five CSVs. It downloads automatically with
  `kagglehub.dataset_download("arnabbiswas1/microsoft-azure-predictive-maintenance")`; no account needed.
* **Code:** three helper files in `src/`. Run notebooks 01–06 in order; 01–05 each save their results to
  `outputs/` for this notebook (appendix B). Machine split: seed 7, stratified by model.

<a id="s5"></a>
## 5 · Main design choices
*Brief: your main design choices (H, W, thresholds, how you defined normal).*

| Choice | What I picked | In plain words |
|---|---|---|
| **How rules were ranked** | one score, F1 | high only when most breakdowns are caught *and* most alarms are real (5.1) |
| **When an alarm counts** (H and W) | an alarm 48 to 12 hours before a breakdown (H = 12, W = 36) | look back as far as the warning sign goes, and keep half a day free to act (5.2) |
| **What "normal" means** | hours far from any breakdown | not in the 48 h before one, not in the day after a repair, not in the first day of data (5.3) |
| **How abnormal counts as abnormal** (thresholds) | 3.09 "typical wobbles" from that machine's own normal | a line random noise crosses only 1 hour in 1,000 (5.4) |
| **What counts as a breakdown** | any part failing; two parts failing on the same machine in the same hour count as one | as the brief defines it; each part was also run on its own |

<a id="s5-1"></a>
### 5.1 · How rules were ranked: one score, F1
**What F1 is.** Every alarm rule is judged on two questions:
1. **Of the breakdowns, how many did it warn about?** (*coverage*, "caught")
2. **Of its alarms, how many were real?** (*precision*, "real")

F1 rolls the two into one score from 0 to 1. Unlike a plain average, it is **pulled down toward the weaker of the
two**. Think of a student who has to pass two exams: scoring 98 on one and 28 on the other doesn't average out to a
pass. F1 is only high when **both** are high.

**Why it's the right yardstick here.** A plant manager needs both answers to be "most": most breakdowns caught,
and most alarms worth walking over for. Each of the brief's three numbers can be gamed on its own:
* **Coverage alone** rewards an alarm that's always on: it "catches" everything, but the crew soon ignores it.
* **Precision alone** rewards firing only on the most obvious cases, missing most breakdowns.
* **Lift alone** rewards a rule that almost never fires: one lucky alarm gives a huge lift.

F1 can't be gamed either way, so it's the one number used to rank all ten ideas and to choose the window. Lift is
still checked (every kept rule clears the brief's bar of 2), just not used to rank.

The chart shows three of the ideas side by side:

In [6]:
# F1 explained with three real ideas (holdout machines): caught (coverage), real (precision), and their F1.
pick = [("C05", "C05 · watch all four sensors"), ("C06", "C06 · any error code"), ("C09", "C09 · matched signature")]
byid = {c["cid"]: c["hold"] for c in K["concepts"]}
fig = go.Figure()
for key, name, color in [("coverage", "breakdowns caught (coverage)", GREY), ("precision", "alarms that were real (precision)", DARK),
                         ("f1", "F1: high only if both are high", BLUE)]:
    vals = [byid[c][key] for c, _ in pick]
    fig.add_bar(x=[n for _, n in pick], y=vals, name=name, marker_color=color,
                text=[f"{v:.0%}" if key != "f1" else f"{v:.2f}" for v in vals], textposition="outside", hoverinfo="skip")
look(fig, "F1 rewards the rule that is good at both, not the one that is great at one", 420)
fig.update_layout(barmode="group", bargap=0.25)
fig.update_yaxes(range=[0, 1.12], tickformat=".0%", title=None)

* **C05** catches 98% of breakdowns, but only 28% of its alarms are real. A plain average would flatter it (63%);
  F1 gives it **0.44**.
* **C06** is right more often (64%), but misses about half the breakdowns: **0.55**.
* **C09** is high on both (99% caught, 90% real): **0.94**.

<a id="s5-2"></a>
### 5.2 · When an alarm counts: the 48-hour window (H and W)
**In plain words:** each part's sensor shifts about 48 hours before it breaks, in nearly every failure on every
machine, so the window looks back 48 hours.
An alarm in the last 12 hours is too late to do anything, so it only counts as a catch if it comes at least 12
hours ahead (H = 12). That leaves 36 hours in which an alarm counts (W = 36).

The timeline shows the window. The chart under it is where the 48 hours comes from.

<div class="timeline">
<b>When does an alarm count as a catch?</b>
<div class="svgbox"><svg viewBox="0 0 760 200" width="100%" role="img" aria-label="Timeline of the 72 hours before a breakdown: the sensor shifts 48 hours before, the error code appears 24 hours before. Alarms count between 48 and 12 hours before; the last 12 hours are kept free to act.">
<rect x="140" y="30" width="460" height="110" fill="#e3f0f1"/>
<rect x="600" y="30" width="115" height="110" fill="#f6ead6"/>
<text x="370" y="50" text-anchor="middle" fill="#0e6170" font-weight="600">Alarm counts as a catch (W = 36 h)</text>
<text x="657" y="50" text-anchor="middle" fill="#b76e00" font-weight="600">Time to act</text>
<text x="657" y="68" text-anchor="middle" fill="#b76e00" font-weight="600">(H = 12 h)</text>
<line x1="30" y1="140" x2="730" y2="140" stroke="#dfe3e1" stroke-width="2"/>
<path d="M30 118 H140 V84 H715" fill="none" stroke="#1d2327" stroke-width="3"/>
<text x="36" y="110" fill="#5d6870">sensor normal</text>
<text x="150" y="76" fill="#5d6870">sensor reads off</text>
<circle cx="370" cy="84" r="8" fill="#0e6170"/>
<text x="382" y="106" fill="#1d2327" font-weight="600">part's own error code</text>
<line x1="715" y1="24" x2="715" y2="140" stroke="#b03a3a" stroke-width="3"/>
<text x="715" y="18" text-anchor="middle" fill="#b03a3a" font-weight="600">Breakdown</text>
<text x="30" y="162" text-anchor="middle" fill="#5d6870">72 h</text>
<text x="140" y="162" text-anchor="middle" fill="#5d6870">48 h before</text>
<text x="370" y="162" text-anchor="middle" fill="#5d6870">24 h before</text>
<text x="600" y="162" text-anchor="middle" fill="#5d6870">12 h before</text>
<text x="715" y="162" text-anchor="middle" fill="#5d6870">0</text>
<text x="370" y="190" text-anchor="middle" fill="#5d6870">hours before the breakdown</text>
</svg></div>
</div>

In [7]:
# The precursor, as notebook 01 drew it (section 6): each part's own signal, averaged over all its failures, for the
# week before the failure. Drawn on top: where the signal shifts, and the scoring window that follows from it.
fig = notebook_figure("01_dataset_and_parameters", trace="comp1: volt")
fig.layout.shapes, fig.layout.annotations = (), ()
fig.add_vrect(x0=48, x1=12, fillcolor=BLUE, opacity=0.10, line_width=0)
fig.add_vrect(x0=12, x1=0, fillcolor=ORANGE, opacity=0.12, line_width=0)
fig.add_vline(x=48, line=dict(color="#1d2327", width=2, dash="dash"))
for x, text, color in [(30, "<b>W = 36</b>: alert counts", BLUE), (6, "<b>H = 12</b>", ORANGE)]:
    fig.add_annotation(x=x, y=1.0, yref="paper", yanchor="bottom", text=text, showarrow=False,
                       font=dict(size=12, color=color))
fig.add_annotation(x=48, y=0.08, yref="paper", text="<b>shift starts 48 h before failure</b>", showarrow=False,
                   xanchor="right", xshift=-8, font=dict(size=12))
fig.add_annotation(x=110, y=0.8, yref="paper", text="flat: no warning sign more than 48 h ahead.<br>"
                   "Blue: an alert here catches the failure (W = 36).<br>Orange: the last 12 h, kept free to act (H = 12).", showarrow=False,
                   font=dict(size=12, color="#5d6870"))
look(fig, "The signal shifts exactly 48 h before failure, so the window looks back 48 h: H + W = 12 + 36", 440)
fig.update_xaxes(autorange="reversed", title="hours before failure", dtick=24)
fig.update_yaxes(title="distance from normal (0 = normal)", zeroline=True, zerolinecolor="#1d2327")
fig.update_layout(margin=dict(t=95))

**How to read this chart.** Each line is one part's own sensor, averaged over every time that part failed and
lined up so the failure is at 0 on the right.
* **The left-to-right axis** is hours before the failure: a week ago on the left, the failure on the right.
* **The up-and-down axis** is how far the sensor is from that machine's normal reading. **0 means normal.** One
  step up or down equals one "typical wobble": how much that sensor normally bounces around on a good day (a
  *standard deviation*). So **+2** means the reading sits about two typical wobbles above normal, and **−1.5** means
  it has dropped well below normal (rotation slows down as a part fails).
* **One machine or all of them?** All of them. Each line averages every failure of that part across all 100
  machines (for example, all 252 rotation failures, spread over 92 machines). Averaging smooths out the hourly
  noise, which is why the lines look so clean. But it isn't just an average effect: when I checked failures one by
  one, 92–98% show the same clear jump, and the typical jump starts 47–48 hours before the failure for every part.
* **Why not plain units?** The four sensors measure different things (volts, rpm, pressure, vibration). Measuring
  each one in "typical wobbles" puts them on one scale, so they can share a chart. It's like a body temperature:
  98.6°F is normal and it wobbles by about half a degree, so 99.6°F is "two wobbles up", a real fever.

### Why H = 12, W = 36
* **W = 36 covers the whole warning sign.** Each part's sensor shifts 48 hours before it fails, and its error code
  lands 24 hours before. A window from 48 to 12 hours before the failure contains both.
* **H = 12 leaves half a day to act.** With no buffer (H = 0), an alarm one hour before a failure would count as a
  catch, which leaves no time to do anything. Twelve hours is about one maintenance shift: enough for the crew on
  duty to diagnose, get the part and repair. Reliability practice sets the warning by the time needed to respond
  (the *net P-F interval* in [reliability-centred maintenance](https://reliabilityweb.com/articles/entry/rcm-on-condition-task-interval-determination);
  [ISO 13381-1](https://www.sis.se/api/document/preview/905442/) bases alarm limits on maintenance lead time).
  For a real plant, H should come from its own repair and spare-part times.

The grids below show what every other split would have scored.

#### Choosing the window: every option tried
Each square is one possible window. **Down the side** is H, the hours kept free to act. **Across the top** is W, the
hours in which an alarm counts. The number in each square is the F1 score ([section 5.1](#s5-1)): darker is better.
Only the **outlined squares** reach back exactly 48 hours, where the warning sign starts, so those are the real
options. The **red square** is the one I chose (H = 12, W = 36).

The first grid is where the choice was made, on the 50 training machines ([section 3](#s3-1) explains the split):

In [8]:
# The window grid, twice. Section 4.1 fixes H + W = 48, so only the outlined cells (windows reaching back exactly 48 h)
# are candidates; the rest are faded. Both grids share one colour scale (0 to 1) so they can be compared cell by cell.
def window_heatmap(G, title):
    Hs, Ws = list(G.index), list(G.columns)
    fig = go.Figure(go.Heatmap(z=G.values, x=[str(w) for w in Ws], y=[str(h) for h in Hs], colorscale="Blues",
                               zmin=0, zmax=1, text=G.values, texttemplate="%{text:.2f}", colorbar=dict(title="F1"),
                               hovertemplate="H = %{y}, W = %{x}: F1 %{z:.3f}<extra></extra>"))
    for yi, h in enumerate(Hs):
        for xi, w in enumerate(Ws):
            box = dict(x0=xi - 0.5, x1=xi + 0.5, y0=yi - 0.5, y1=yi + 0.5, xref="x", yref="y")
            if h + w != 48:
                fig.add_shape(type="rect", fillcolor="white", opacity=0.78, line_width=0, **box)
            else:
                fig.add_shape(type="rect", line=dict(color="red" if h == 12 else "#1d2327", width=4 if h == 12 else 2), **box)
    notes = {(0, 48): "no time to act", (12, 36): "chosen", (24, 24): "edge: error code<br>at 24 h",
             (36, 12): "closes before<br>the error code"}
    for (h, w), text in notes.items():
        fig.add_annotation(x=Ws.index(w), y=Hs.index(h), text=text, showarrow=False, xanchor="left", xshift=34,
                           font=dict(size=11, color="red" if h == 12 else "#1d2327"), bgcolor="rgba(255,255,255,0.85)")
    look(fig, title, 520)
    fig.update_xaxes(title="W: hours in which an alarm counts", type="category", range=[-0.5, len(Ws) - 0.5])
    fig.update_yaxes(title="H: hours kept free to act", type="category", range=[len(Hs) - 0.5, -0.5])
    return fig

# 1) DEV machines (notebook 04): the leading rule re-tuned for every window. This is the grid the window was chosen on.
G_dev = pd.DataFrame(N4["hw_grid"]).set_index("H")
G_dev.columns = G_dev.columns.astype(int)
window_heatmap(G_dev, "1 · Training machines: where the window was chosen")

The second grid is the check: the same rule, held fixed, scored on the 50 test machines it never saw. Nothing was
chosen here.

In [9]:
# 2) HOLDOUT machines: the kept C09 (tuned for H = 12, W = 36, then held fixed) scored at every window. These scores
# were computed by the website build for the Rule explorer (site/build_site.py -> docs/data/rules.json), from the saved
# rules; they were never used to choose anything.
R = json.load(open(ROOT / "docs" / "data" / "rules.json"))
cells = R["data"][R["tunings"][0]]["any"]
G_hold = pd.DataFrame([[cells[f"{h},{w}"]["C09"]["f1"] for w in G_dev.columns] for h in G_dev.index],
                      index=G_dev.index, columns=G_dev.columns)
window_heatmap(G_hold, "2 · Test machines (check only): does the choice hold up?")

**What the two grids show:**
* **A buffer of up to 12 hours costs almost nothing.** H = 0 and H = 12 are within 0.01 on both grids (0.945 vs
  0.936 on training machines, 0.947 vs 0.941 on test machines). So I took the 12 hours to act.
* **Waiting until 24 hours is too close to the edge.** That's exactly when the error code lands. On the test
  machines the score drops to 0.84.
* **Past 24 hours, everything collapses.** The window closes before the error code arrives, so C09 can't fire in time.
* **The two grids agree,** so a choice made on training machines carries over to machines it never saw.

<a id="s5-3"></a>
### 5.3 · What "normal operation" means
**Normal** hours are hours far from any breakdown: not in the 48 hours before one, not in the 24 hours after a
breakdown or repair (the crew is at the machine and readings reflect the restart), and not in the first 24 hours of
data (the averages need a day of history). Every threshold is set from normal hours only, as the brief requires.

<a id="s5-4"></a>
### 5.4 · How abnormal counts as abnormal (thresholds)
**In plain words:** each machine is compared with **its own** normal readings, like comparing your temperature with
your own normal rather than everyone's. A sensor counts as "off" once it's more than **3.09 typical wobbles** from
that normal. Random noise crosses that line only **1 hour in 1,000**.

**I didn't pick 3.09 by hand.** Five lines were tried, from loose to strict, and C09 did best with this one
([appendix A.3](#appx-a3)). Even the strictest line can't remove the false "fevers": they're real shifts, not noise.
That's why the error code is needed.

**It matches long-standing factory practice.** Since 1935, British quality-control charts have used an "action
limit" set so chance alone crosses it only 1 time in 1,000, which is exactly 3.09 typical wobbles
([source](https://www.encyclopedia.com/science/dictionaries-thesauruses-pictures-and-press-releases/shewhart-walter-andrew)).
C09's tuning landed on the same line on its own.

<a id="s6"></a>
## 6 · What I would do next
*Brief: if you run out of time, what you'd do next.*

* **Ask the plant about the false "fevers".** If a crew fixed something before it broke, some "false alarms" were
  really saves.
* **Score in money.** Once we know what a false alarm costs versus a missed failure, rank rules by money saved.
* **Test on real data.** Here the warning signs land at exactly 48 and 24 hours. A real plant's history would show
  how much warning there really is, and set the window from that.
* **Cut repeat alarms.** C09 often fires about twice per failure. Muting it for a day after the first alarm should
  roughly halve the noise.
* **Push the ML further:** one model per part, and re-running the training/test split many times to check the
  choice is stable.

<a id="s7"></a>
## 7 · How I used AI, and where I corrected it
*Brief: if you used AI tools, transcripts and notes on corrections.*

I used Claude as a coding assistant. I directed the analysis and made the key calls; it wrote most of the code to my
instructions. These are the places where I caught it going wrong (full list in the
[README](https://github.com/avaitt/Oden-Take-Home-Assessment#where-the-ai-assistant-had-to-be-guided)):

| What the AI did | What I changed |
|---|---|
| Built the test machines' thresholds from their own breakdown dates, and measured which signals matter on test machines | Caught the leak. Every choice now uses training machines only |
| Chose H = 0, so a 1-hour warning counted as a catch | Required time to act, which led to H = 12, W = 36 |
| Set a 24-hour lead on an untested assumption | Questioned it; an hour-by-hour audit showed it had no basis |
| Claimed the 12-hour buffer "costs ~0.01 F1", having checked only training machines | Tested it on the test machines in the Rule explorer: H = 12 held up (0.941 vs 0.947), but H = 24 dropped to 0.84, so the margin matters |
| Explained things with jargon ("roll6", "robust", "alert lines") and confusing tables | Had them rewritten in plain language |

<a id="s8"></a>
## 8 · How this mirrors Oden's Agentic Root Cause Analysis workflow
*Brief (background): build a small version of Oden's root-cause workflow.*

Each of the five steps in Oden's [Agentic Root Cause Analysis](https://engineering.oden.io/blog/agentic-root-cause-analysis) post has a smaller counterpart here:

| Oden's workflow | This project |
|---|---|
| **1. Build operating context** (line diagrams, process knowledge, operator interviews) | Explored all five data files and checked hour by hour how failures are logged (notebook 01) |
| **2. Reduce the search space with ML** (gradient-boosted classifiers, feature importances) | Trained ML models to see which signals matter most, used for ideas rather than as the alarm (notebook 03) |
| **3. Generate hypotheses** that tie a signal pattern to a physical mechanism and a testable statistic | Ten candidate warning signs, each with a physical reason and a fixed yes/no rule (notebook 02) |
| **4. Filter by lift analysis**: lift, precision, and permutation tests against random nominal windows | Scored every idea on lift, precision and coverage, and checked each one beats scrambled copies of itself; thresholds set from normal running only (notebook 04) |
| **5. Root-cause families, playbooks, alerts** | One family per part: C09 (own signal + own error code) is the alarm to deploy; C10 (drift on an aging part) is an optional earlier watchlist. Playbooks would come next, with plant input |

Two points from Oden's post show up in this data too:
* **Some "false alarms" may be saves.** Operators sometimes fix a problem before it fails. The ~470 false "fevers"
  per part could include some of those, which is why the error code is needed to confirm.
* **Simple, explainable alarms win.** C09 is one easy-to-explain pattern, applied to each part: its own sensor is
  off *and* its own error code fired.

<a id="s9"></a>
## 9 · Impact: what C09 gives you that you didn't have before

**Before this analysis**, a plant with this data could wait for breakdowns, alarm on sensors that cry wolf, or
replace parts on a schedule. None of them works well:

<div class="compare">
<div><h4>Without C09</h4><ul>
<li>Breakdowns arrive as surprises</li>
<li>Sensor alarms: about 1 in 4 real, so crews tune them out</li>
<li>Replacing parts on a schedule: no real warning (C07)</li>
<li>An alarm says "something's off", not what</li>
<li>Black-box ML: right 46% of the time</li>
</ul></div>
<div class="after"><h4>With C09</h4><ul>
<li><b>99%</b> of breakdowns flagged ahead of time (346 of 350)</li>
<li><b>9 in 10</b> alarms real; under 2 false alarms per machine per year</li>
<li>The alarm names <b>which part</b> is failing</li>
<li>One sentence anyone on the floor can check</li>
</ul></div>
</div>

**What changes on the floor:**
* **About a day of warning, so repairs can be planned.** C09 fires about 24 hours before a failure, and only alarms
  at least 12 hours ahead count. That's a full shift to diagnose, get the part and fix it during a planned stop
  instead of an unplanned one.
* **Alarms worth answering.** On the 50 test machines, C09 raised just **85 false alarms in a year, across all 50
  machines**, while warning about 346 breakdowns. A crew can afford to respond to every one.
* **It points at the part.** Because C09 pairs each part's own sensor with its own error code, the alarm says
  *which* part (say, comp3, pressure), so the crew brings the right spare.
* **Nothing new to install.** It uses the sensors and error log the plant already has.
* **Easy to trust.** "This part's sensor has been off for hours, and its own fault code just fired" is something an
  operator can check on screen. There's no model to maintain or explain.

**What it's worth** depends on the plant's cost of an hour of downtime, which this data doesn't include. Once a
plant shares that, the next step is to put a dollar figure on it ([section 6](#s6)). And since the data is
simulated, real-world numbers will be lower. What carries over is the method: find each part's own sensor and fault
code, pair them, and prove it on machines the rule has never seen.

---
<a id="appx"></a>
## Appendix
The technical detail behind sections 1–9: <a href="#appx-a">A · key terms and exact definitions</a>,
<a href="#appx-b">B · how the project fits together</a>, <a href="#appx-c">C · what each notebook found</a>.

<a id="appx-a"></a>
## Appendix A · Key terms and exact definitions
### A.1 · Key terms
| Term | Meaning |
|---|---|
| **Event** | a component failure. Failures on the same machine at the same hour count as one event (the brief's rule). "Any failure" is the main target; each part (comp1–comp4) is also run on its own. |
| **Precursor / rule** | a pattern that tends to come before a failure, written as a simple yes/no rule checked every hour ("alarm" or "no alarm"). |
| **H and W** | for a failure at time T, the failure counts as *caught* if an alarm falls between T − (H + W) and T − H. Here **H = 12, W = 36**: an alarm between 48 and 12 hours before the failure. |
| **Normal operation** | hours far from any failure: not in the 48 h before one, not in the 24 h after a failure or repair, and not in the first 24 h of data. |
| **Lift** | how much more often a rule fires before failures than in normal operation (2 = twice as often). |
| **Precision** | of the alarms a rule raises, the share followed by a failure within 48 hours (H + W). |
| **Coverage** | of the failures, the share that had at least one alarm in their window. |
| **F1** | the single score used to pick rules: `2 × precision × coverage / (precision + coverage)`. It is high only when *both* are high, so it can't be gamed by never firing or always firing. |
| **Training / test machines** (dev / holdout) | the 100 machines are split 50/50. The rules are **built and tuned on the training (dev) machines** and **scored on the test (holdout) machines**, which play no part in building them. |
| **Drift** | a sensor moving away from *that machine's own* normal level and **staying there**, not just blipping. Like a person's temperature: one high reading is noise; running a fever all day is worth noticing. Each part's own sensor drifts in the two days before it fails. |
| **Look-alike drift** | a drift that looks just like the ones before a failure, but no failure follows: the machine has an off day and settles back (~470 per part per year). |

<a id="appx-a2"></a>
### A.2 · The three rules, written out
Each rule is checked **every hour on every machine**, and raises an alarm when **both** of its conditions hold.
* **6 h average:** the mean of a sensor's last 6 hourly readings.
* **Typical wobble:** how much a sensor normally bounces around on that machine (a standard deviation). Each
  machine is compared with its own normal ([section 5.4](#s5-4) explains the 3.09 line).
* **Each part's own sensor and error code(s):**

| Part | Sensor (direction of a fault) | Own error code(s) |
|---|---|---|
| comp1 | voltage (rises) | error1 |
| comp2 | rotation (falls) | error2 or error3 |
| comp3 | pressure (rises) | error4 |
| comp4 | vibration (rises) | error5 |

**C09 · Matched signature** (the alarm to act on). For any one part:
1. its sensor's 6 h average is **more than 3.09 typical wobbles away from that machine's normal**, in the fault direction
   (on a typical machine: voltage above ~190 vs ~170 usually, rotation below ~383 vs ~449, pressure above ~113 vs
   ~100, vibration above ~47 vs ~40). 3.09 is the 1-in-1,000 line ([section 5.4](#s5-4)), **and**
2. that part's **own error code** was logged **at least once in the last 24 hours**.

**C08 · Drift + any error** (the fallback):
1. **any** of the four sensors' 6 h average is **more than 3.72 typical wobbles away from that machine's normal**, in its
   fault direction, **and**
2. **any** error code was logged **at least once in the last 48 hours**.

**C10 · Drift on an aging part** (the early warning). For any one part:
1. its sensor's 6 h average is **more than 3.72 typical wobbles away from that machine's normal**, in the fault
   direction, **and**
2. the part has gone **longer since its last replacement than usual**: about 33 days for comp1, 31 for comp2,
   34 for comp3 and 33 for comp4.

<a id="appx-a3"></a>
### A.3 · The five sensor lines tried
Each idea was tried with five lines, from loose to strict ([section 5.4](#s5-4)):

| How strict | Typical wobbles from normal | Random noise crosses it | Normal hours that actually cross it | Used by |
|---|---|---|---|---|
| loosest | 1.64 | 1 hour in 20 | 6.8% | |
| | 2.33 | 1 in 100 | 3.3% | |
| | 2.58 | 1 in 200 | 2.8% | |
| **chosen** | **3.09** | **1 in 1,000** | **2.2%** | **C09** |
| strictest | 3.72 | 1 in 10,000 | 1.7% | C08, C10 |

**How to read it.** Going down, the line gets 500 times stricter for random noise, but the share of normal hours
crossing it stops falling at about 2%. Those 2% aren't noise: they're the false "fevers", real shifts that no
failure follows. So the sensor line alone can't be the alarm. That's the error code's job.

<a id="appx-a4"></a>
### A.4 · Every idea at a glance
Each dot is one of the ten ideas at its best, on the test machines: **further right** catches more breakdowns,
**higher up** raises fewer false alarms. Only C09 is high on both. Hover for details.

In [10]:
# Every idea's best version on the holdout machines: failures caught (coverage) against alarms that were real
# (precision). Grey curves join points of equal F1; top right is best.
fig = go.Figure()
for f1 in (0.2, 0.4, 0.6, 0.8, 0.9):
    p = np.linspace(f1 / (2 - f1), 1, 200)
    fig.add_scatter(x=f1 * p / (2 * p - f1), y=p, mode="lines", line=dict(color="#dfe3e1", width=1),
                    hoverinfo="skip", showlegend=False)
    fig.add_annotation(x=f1 / (2 - f1), y=1.0, text=f"F1 {f1}", showarrow=False, yshift=9,
                       font=dict(size=10, color="#8a9499"))
groups = [("kept: C09, the rule to deploy", lambda c: c["cid"] == "C09", BLUE, 16),
          ("kept: C08, C10", lambda c: c["status"] == "kept" and c["cid"] != "C09", DARK, 11),
          ("rejected", lambda c: c["status"] != "kept", GREY, 10)]
for name, keep, color, size in groups:
    cs = [c for c in K["concepts"] if keep(c)]
    fig.add_scatter(x=[c["hold"]["coverage"] for c in cs], y=[c["hold"]["precision"] for c in cs],
                    mode="markers+text", text=[c["cid"] for c in cs],
                    textposition=["bottom center" if c["cid"] == "C01" else "top center" for c in cs], name=name,
                    marker=dict(size=size, color=color, line=dict(color="white", width=1.5)),
                    customdata=[[rule_name(c), c["hold"]["f1"]] for c in cs],
                    hovertemplate="%{text} · %{customdata[0]}<br>coverage %{x:.0%} · precision %{y:.0%}"
                                  " · F1 %{customdata[1]:.2f}<extra></extra>")
look(fig, "All ten ideas on machines they never saw: only C09 is high on both", 460)
fig.update_xaxes(title="coverage: share of failures caught", tickformat=".0%", range=[-0.03, 1.05])
fig.update_yaxes(title="precision: share of alarms that were real", tickformat=".0%", range=[0, 1.08])

<a id="appx-b"></a>
## Appendix B · How the project fits together, and where the numbers come from
**Two kinds of files:**
* **`src/` (three helper files)** hold every function: `data.py` (settings, loading, windows), `detectors.py`
  (the rules and the ten candidates), `evaluation.py` (metrics, the selection protocol, audits, ML benchmark,
  and saving results).
* **Notebooks 01–06.** Run them **in order**. Each of 01–05 computes its own analysis live from the raw data
  and, in its last cell, saves its key results to `outputs/`:

| Run | Computes | Saves |
|---|---|---|
| 01 Dataset and parameters | the dataset profile, the timestamp audit, look-alike drifts, confidence | `outputs/01_dataset_and_parameters.json` |
| 02 Candidates | the ten candidates and their variants | `outputs/02_candidates.json` |
| 03 ML benchmark (~2 min) | three ML models, trained and scored | `outputs/03_ml_benchmark.json` |
| 04 Test and select (~10 min) | the full selection protocol for all five event types, and the H × W grid | `outputs/04_test_and_select.json` |
| 05 Audit | the audits of the detectors 04 kept (it reads 04's file for which ones) | `outputs/05_audit.json` |
| 06 Write-up (this one) | nothing: it reads the five files above, plus the holdout window grid the website build saved for the Rule explorer (`docs/data/rules.json`) | — |

**Why save results at all?** So this write-up can summarise everything without re-running ~15 minutes of
analysis, and so every number quoted here comes from the notebook that produced it. Each file records the
settings it was made with (H, W, seed) and the time it ran.

**Guarding against out-of-date files.** The table at the top of this notebook lists each file with the time
it ran. It says "re-run" if the code in `src/` changed afterwards, or if notebook 04 was re-run after notebook
05 (05 audits 04's detectors), and it warns if the notebooks used different H and W settings.

When each notebook last ran, and whether its saved results are still up to date:

In [11]:
display(check_results()[["ran_at", "status"]])   # src/evaluation.py §9

,ran_at,status
notebook,,
01_dataset_and_parameters,2026-11-02 04:06:47,up to date
02_candidates,2026-11-02 04:06:50,up to date
03_ml_benchmark,2026-11-02 04:08:07,up to date
04_test_and_select,2026-11-02 04:17:07,up to date
05_audit,2026-11-02 04:17:22,up to date


<a id="appx-c"></a>
## Appendix C · What each notebook found

Each subsection gives the notebook's key conclusions, then the saved numbers behind them.

### C.1 · Notebook 01: dataset and parameters
| Characteristic | Finding |
|---|---|
| Events | 761 failure rows → **719 events** (same machine + hour merged); 131–259 per part |
| Sampling and gaps | hourly, 100 machines × 8,761 h (Jan 1, 2015 to Jan 1, 2016; maintenance log from mid-2014); no gaps, no missing values |
| Clustering | **not a factor here:** only 3 of 719 events come within a week of the previous one on the same machine, so clustering in time changes nothing in the analysis. Failures do vary *by machine* (0–19 each), which the confidence intervals handle by resampling whole machines |
| Operating regimes | effectively one: the four machine models run at the same signal levels, and machines never stop |
| Counters vs. live readings | the four sensors (volt, rotate, pressure, vibration) are live readings; the only counter is "hours since maintenance", which I build from the maintenance log |
| Timestamps | failures are logged in a daily 06:00 batch, but an hour-by-hour check finds no post-failure data before the stamp |
| Where the warning signs are | each part has one signature signal (comp1 → volt, comp2 → rotate, comp3 → pressure, comp4 → vibration). It shifts exactly 48 h before failure; the part's own error code lands exactly 24 h before |
| Key property | ~470 look-alike drifts per part never end in a failure, capping a signal-only rule at 21–34% precision; the error code is in 96–100% of real drifts but only 14–18% of look-alikes |
| Confidence | 350 scored holdout events for "any failure" (of 363; ±5 points), 65 for comp3 (±12); the data is synthetic, so treat absolute numbers as optimistic |

This notebook's two key charts are in the main write-up: why the error code matters (<a href="#s1">section 1</a>) and the 48-hour window (<a href="#s5-2">section 5.2</a>).

### C.2 · Notebook 02: candidate precursors
**Where the ten ideas come from.** Before a failure, the dataset offers three kinds of evidence: four live
sensors, an error-code log, and a maintenance log (from which I build each part's age). I listed the ideas in
two rounds. First, what each kind of evidence could show by itself (C01–C07). These are the ideas anyone would
write down before looking at the data. Then, combinations suggested by what notebook 01 found in the data and
what the ML models in notebook 03 leaned on (C08–C10). Keeping the obvious ideas in the race shows whether the
data-driven ones really add anything (section 2). This is step 3 of Oden's workflow, "generate hypotheses"
(section 8).

| ID | Idea | Why it is on the list |
|---|---|---|
| C01–C04 | Voltage creeps up · Rotation sags · Pressure builds · Vibration rises | **Domain knowledge, one per sensor.** Each is a classic failure mode of rotating equipment: an electrical fault disturbs the voltage, friction slows the shaft, a blockage raises pressure, bearing wear shakes the machine. They are the simplest possible precursors and the baseline the others must beat. |
| C05 | Any signal drifts | "Any failure" doesn't say which part will fail, so one alarm watches all four sensors at once. |
| C06 | Any error code | **The second source of evidence.** Controllers log a fault code when a part misbehaves, often before it breaks. |
| C07 | Overdue maintenance | **The third source.** Parts wear out with time since replacement. I included it knowing that a counter can quietly use the future, so the gate has a plausible idea to test. It fails it. |
| C08 | Drift + any error | **From the data:** each part's signal drifts ~470 times a year without a failure, but an error code shows up in 96–100% of real drifts and only 14–18% of look-alikes (notebook 01). **From the ML models:** they lean on smoothed signals and error counts (notebook 03). Both point to a drift confirmed by an error. |
| C09 | Matched signature | Notebook 01 also found that each part has its **own** sensor and its **own** error code, so C08 is sharpened to require both for the **same** part. |
| C10 | Drift on an aging part | A drift on a part long past its last replacement is more likely to be real. Unlike C09, it doesn't wait for the error code, so it could warn earlier. |

**Where "655 variants" comes from.** An idea is not one rule. "Voltage creeps up" can be checked on the raw
hourly reading or a smoothed one, against a fleet-wide or a per-machine usual level, at a strict or a loose
cut-off. Judging an idea by its first draft could reject a good idea because of a bad setting. So each idea is
built in every reasonable setting, and judged by its best version on the dev machines:

| Ideas | Settings tried | Variants |
|---|---|---|
| C01–C05 (sensors) | 5 ways to read the signal (hourly reading, held 3+ h, 6 h average, 24 h average, CUSUM) × 4 usual levels (fleet, machine model, machine, robust per machine) × 5 cut-offs (P95 to P99.99 of normal operation), less the 5 CUSUM + robust combinations, which don't work (CUSUM sits at 0 most of the time) | 95 each, 475 in all |
| C06 (errors) | error count over the last 24 h or 48 h × fleet or per-machine level × 5 cut-offs (P80 to P99.5) | 20 |
| C07 (part age) | 4 usual levels × 5 cut-offs | 20 |
| C08 | 6 h or 24 h average × 2 per-machine levels × 5 sensor cut-offs × 2 error cut-offs | 40 |
| C09 | 6 h or 24 h average × 2 per-machine levels × 5 sensor cut-offs × error look-back of 24 h or 48 h | 40 |
| C10 | 6 h or 24 h average × 2 per-machine levels × 5 sensor cut-offs × 3 part-age cut-offs (P50, P75, P90) | 60 |
| **All ten** | | **655** |

Every cut-off comes from normal-operation hours only. Trying 655 versions also makes a lucky winner possible,
which is why every idea must pass the gates (section 2) and why notebook 05 checks for luck (appendix C.5).

In [12]:
# Saved by notebook 02: the ten candidates, with what each one alerts on in plain words.
WHAT = {
    "C01": "a machine's voltage rises well above its normal level (an electrical fault)",
    "C02": "a machine's rotation speed drops well below its normal level (friction or wear)",
    "C03": "a machine's pressure rises well above its normal level (a blockage)",
    "C04": "a machine's vibration rises well above its normal level (bearing wear)",
    "C05": "any of the four sensors moves well past its normal level",
    "C06": "the machine logs an error code, any code",
    "C07": "a part has gone much longer than usual since its last replacement",
    "C08": "a sensor drifts AND an error code (any code) was logged recently",
    "C09": "a part's own sensor drifts AND that same part's own error code was logged recently",
    "C10": "a part's own sensor drifts AND that part is older than usual since its last replacement",
}
T = pd.DataFrame(N2["candidates"])[["id", "title", "family", "variants"]]
T["title"] = [NAME.get(i, t) for i, t in zip(T["id"], T["title"])]
T.insert(2, "what it alerts on", T["id"].map(WHAT))
T.style.hide(axis="index").set_properties(subset=["what it alerts on"], **{"text-align": "left"})

id,title,what it alerts on,family,variants
C01,Voltage creeps up,a machine's voltage rises well above its normal level (an electrical fault),telemetry,95
C02,Rotation sags,a machine's rotation speed drops well below its normal level (friction or wear),telemetry,95
C03,Pressure builds,a machine's pressure rises well above its normal level (a blockage),telemetry,95
C04,Vibration rises,a machine's vibration rises well above its normal level (bearing wear),telemetry,95
C05,Any signal drifts,any of the four sensors moves well past its normal level,combo,95
C06,Any error code,"the machine logs an error code, any code",errors,20
C07,Overdue maintenance,a part has gone much longer than usual since its last replacement,counter,20
C08,Drift + any error,a sensor drifts AND an error code (any code) was logged recently,combo,40
C09,Matched signature,a part's own sensor drifts AND that same part's own error code was logged recently,combo,40
C10,Drift on an aging part,a part's own sensor drifts AND that part is older than usual since its last replacement,combo,60


**From ideas to three rules.** Part A shows where the ten ideas come from. Part B shows how they were tuned
and selected by a fixed procedure that is the same for every idea. Every choice in B1–B7 uses only the dev machines.
Holdout scores are recorded along the way but never used to choose anything; they are only reported at the end.

<svg viewBox="0 0 720 858" width="100%" style="max-width:720px;display:block;margin:12px 0;font-family:inherit" role="img" aria-label="Flowchart: how the ten ideas came to be, and how they were tuned and selected">
<defs><marker id="arr" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="currentColor" fill-opacity="0.6"/></marker></defs>
<text x="20" y="18" font-size="13" font-weight="700" letter-spacing="0.06em" fill="currentColor" fill-opacity="0.65">A · HOW THE TEN IDEAS CAME TO BE</text>
<rect x="20" y="30" width="440" height="68" rx="8" fill="#3a6ea5" fill-opacity="0.12" stroke="#3a6ea5" stroke-width="1.5"/>
<text x="34" y="52" font-size="15" font-weight="600" fill="currentColor">A1 · Common sense: one idea per kind of evidence</text>
<text x="34" y="72" font-size="12.5" fill="currentColor" fill-opacity="0.78">each sensor on its own (C01–C04), all four at once (C05),</text>
<text x="34" y="90" font-size="12.5" fill="currentColor" fill-opacity="0.78">any error code (C06), a part overdue for replacement (C07)</text>
<line x1="240.0" y1="98" x2="240.0" y2="122" stroke="currentColor" stroke-opacity="0.6" stroke-width="1.5" marker-end="url(#arr)"/>
<rect x="20" y="124" width="440" height="86" rx="8" fill="#3a6ea5" fill-opacity="0.12" stroke="#3a6ea5" stroke-width="1.5"/>
<text x="34" y="146" font-size="15" font-weight="600" fill="currentColor">A2 · What the data showed: combine them</text>
<text x="34" y="166" font-size="12.5" fill="currentColor" fill-opacity="0.78">sensors often drift with no failure; an error code tells the real</text>
<text x="34" y="184" font-size="12.5" fill="currentColor" fill-opacity="0.78">drifts apart (C08). Each part has its own sensor and error code</text>
<text x="34" y="202" font-size="12.5" fill="currentColor" fill-opacity="0.78">(C09). Worn parts fail more, so drift + part age (C10)</text>
<line x1="240.0" y1="210" x2="240.0" y2="234" stroke="currentColor" stroke-opacity="0.6" stroke-width="1.5" marker-end="url(#arr)"/>
<text x="20" y="254" font-size="13" font-weight="700" letter-spacing="0.06em" fill="currentColor" fill-opacity="0.65">B · HOW THE TEN WERE TUNED AND SELECTED (NOTEBOOK 04)</text>
<rect x="20" y="266" width="440" height="50" rx="8" fill="currentColor" fill-opacity="0.05" stroke="currentColor" stroke-opacity="0.45" stroke-width="1.5"/>
<text x="34" y="288" font-size="15" font-weight="600" fill="currentColor">B1 · Ten ideas, C01–C10</text>
<text x="34" y="308" font-size="12.5" fill="currentColor" fill-opacity="0.78">each with a physical story (notebook 02)</text>
<line x1="240.0" y1="316" x2="240.0" y2="340" stroke="currentColor" stroke-opacity="0.6" stroke-width="1.5" marker-end="url(#arr)"/>
<rect x="20" y="342" width="440" height="50" rx="8" fill="currentColor" fill-opacity="0.05" stroke="currentColor" stroke-opacity="0.45" stroke-width="1.5"/>
<text x="34" y="364" font-size="15" font-weight="600" fill="currentColor">B2 · Build 655 variants</text>
<text x="34" y="384" font-size="12.5" fill="currentColor" fill-opacity="0.78">every idea in every reasonable setting (table above)</text>
<line x1="240.0" y1="392" x2="240.0" y2="416" stroke="currentColor" stroke-opacity="0.6" stroke-width="1.5" marker-end="url(#arr)"/>
<rect x="20" y="418" width="440" height="50" rx="8" fill="currentColor" fill-opacity="0.05" stroke="currentColor" stroke-opacity="0.45" stroke-width="1.5"/>
<text x="34" y="440" font-size="15" font-weight="600" fill="currentColor">B3 · Set cut-offs and score on dev</text>
<text x="34" y="460" font-size="12.5" fill="currentColor" fill-opacity="0.78">cut-offs from normal hours only; F1 on the 50 dev machines</text>
<line x1="240.0" y1="468" x2="240.0" y2="492" stroke="currentColor" stroke-opacity="0.6" stroke-width="1.5" marker-end="url(#arr)"/>
<rect x="20" y="494" width="440" height="50" rx="8" fill="currentColor" fill-opacity="0.05" stroke="currentColor" stroke-opacity="0.45" stroke-width="1.5"/>
<text x="34" y="516" font-size="15" font-weight="600" fill="currentColor">B4 · Keep each idea's best variant</text>
<text x="34" y="536" font-size="12.5" fill="currentColor" fill-opacity="0.78">10 tuned rules, one per idea, chosen by dev F1</text>
<line x1="240.0" y1="544" x2="240.0" y2="568" stroke="currentColor" stroke-opacity="0.6" stroke-width="1.5" marker-end="url(#arr)"/>
<rect x="20" y="570" width="440" height="50" rx="8" fill="currentColor" fill-opacity="0.05" stroke="currentColor" stroke-opacity="0.45" stroke-width="1.5"/>
<text x="34" y="592" font-size="15" font-weight="600" fill="currentColor">B5 · Four gates (dev machines)</text>
<text x="34" y="612" font-size="12.5" fill="currentColor" fill-opacity="0.78">not luck · lift ≥ 2 · coverage ≥ 5% · causal check</text>
<line x1="460" y1="595.0" x2="498" y2="595.0" stroke="#d1495b" stroke-opacity="0.8" stroke-width="1.5" marker-end="url(#arr)"/>
<rect x="500" y="570.5" width="216" height="49" rx="8" fill="#d1495b" fill-opacity="0.10" stroke="#d1495b" stroke-dasharray="5 4" stroke-width="1.5"/>
<text x="512" y="592.5" font-size="14" font-weight="600" fill="currentColor">Fails a gate</text>
<text x="512" y="611.5" font-size="11.5" fill="currentColor" fill-opacity="0.78">C07 overdue maintenance</text>
<line x1="240.0" y1="620" x2="240.0" y2="644" stroke="currentColor" stroke-opacity="0.6" stroke-width="1.5" marker-end="url(#arr)"/>
<rect x="20" y="646" width="440" height="50" rx="8" fill="currentColor" fill-opacity="0.05" stroke="currentColor" stroke-opacity="0.45" stroke-width="1.5"/>
<text x="34" y="668" font-size="15" font-weight="600" fill="currentColor">B6 · Rank by dev F1</text>
<text x="34" y="688" font-size="12.5" fill="currentColor" fill-opacity="0.78">skip any rule that is the same alarm as a better one</text>
<line x1="460" y1="671.0" x2="498" y2="671.0" stroke="#d1495b" stroke-opacity="0.8" stroke-width="1.5" marker-end="url(#arr)"/>
<rect x="500" y="646.5" width="216" height="49" rx="8" fill="#d1495b" fill-opacity="0.10" stroke="#d1495b" stroke-dasharray="5 4" stroke-width="1.5"/>
<text x="512" y="668.5" font-size="14" font-weight="600" fill="currentColor">Outranked</text>
<text x="512" y="687.5" font-size="11.5" fill="currentColor" fill-opacity="0.78">C01–C06: real but weaker signals</text>
<line x1="240.0" y1="696" x2="240.0" y2="720" stroke="currentColor" stroke-opacity="0.6" stroke-width="1.5" marker-end="url(#arr)"/>
<rect x="20" y="722" width="440" height="50" rx="8" fill="#2a9d8f" fill-opacity="0.18" stroke="#2a9d8f" stroke-width="1.5"/>
<text x="34" y="744" font-size="15" font-weight="600" fill="currentColor">B7 · Keep the top three</text>
<text x="34" y="764" font-size="12.5" fill="currentColor" fill-opacity="0.78">C09 matched signature · C08 drift + any error · C10 aging part</text>
<line x1="240.0" y1="772" x2="240.0" y2="796" stroke="currentColor" stroke-opacity="0.6" stroke-width="1.5" marker-end="url(#arr)"/>
<rect x="20" y="798" width="440" height="50" rx="8" fill="#2a9d8f" fill-opacity="0.18" stroke="#2a9d8f" stroke-width="1.5"/>
<text x="34" y="820" font-size="15" font-weight="600" fill="currentColor">B8 · Report on holdout</text>
<text x="34" y="840" font-size="12.5" fill="currentColor" fill-opacity="0.78">50 unseen machines; their scores never chose anything</text>
</svg>


#### Precursor vs. variant: what is being tested
A **precursor** is *what* to look for. A **variant** is *how exactly* it is measured.

* **The precursor (the idea)** fixes which evidence is watched (a sensor, an error code, part age, or a
  combination) and what pattern counts as a warning ("this sensor goes abnormally high", "a sensor drifts *and*
  that part's error code appears"). Two different precursors make different claims about what comes before a
  failure.
* **A variant** keeps the idea fixed and turns the knobs on how it is measured:

| Knob | The question it answers | Choices tried |
|---|---|---|
| **Smoothing** | Read the raw reading, or an average? | hourly reading, held 3+ h, 6 h average, 24 h average, CUSUM |
| **Usual level** | "Abnormal" compared with what? | the whole fleet, machines of the same model, this machine's own history (plain or robust) |
| **Cut-off** | How abnormal before it counts? | from the top 5% to the top 0.01% of normal running |
| **Look-back** (error codes, part age) | How recent or how old must the evidence be? | error codes in the last 24 h or 48 h; part age above its P50, P75 or P90 |

So "vibration rises, 6 h average, compared with its own machine, top 1%" and "vibration rises, raw reading,
compared with the fleet, top 5%" are **the same claim measured two ways**, and one may simply be the better
measurement.

**How every variant is tested (the same for all 655):** set its alert line from normal-running hours only → run
it every hour of 2015 on every machine → each off-to-on switch is an alert → score it on the 50 dev machines:
precision (alerts followed by a failure within 48 h), coverage (failures with an alert 48 to 12 h before) and
their F1. Holdout scores are recorded but never used to choose.

**Why the comparison happens in two stages:**
1. **Within each idea**, its variants compete on dev F1 and only the best survives. That is tuning: each idea
   gets its best shot, so a good idea is never rejected for a bad setting.
2. **Between ideas**, only those best versions compete, through the gates and the ranking (steps B5–B7 above).
   Comparing variant A of one idea with variant B of another directly would be unfair, because one could lose
   only because it had worse settings.


### C.3 · Notebook 03: ML benchmark
* Three models (gradient boosting with and without the maintenance counters, and logistic regression), trained
  on dev machines from 32 features and scored exactly like the rules. The best, logistic regression, reaches
  F1 ~0.63, well below C09; section 2.2 compares them and explains why the rules win.
* **What the models lean on:** smoothed signals (6 h and 24 h averages) and each part's own error counts, which
  pointed the candidates toward "smoothed drift + error code".

In [13]:
# Saved by notebook 03: each ML model on the holdout machines.
pd.DataFrame(N3["models"])[["detector", "F1", "F1_CI", "precision", "coverage"]].rename(columns={"F1_CI": "F1 95% interval"}
    ).style.hide(axis="index").format({"F1": "{:.2f}", "precision": "{:.0%}", "coverage": "{:.0%}"})

detector,F1,F1 95% interval,precision,coverage
ML: logistic regression,0.63,"[0.59, 0.66]",46%,97%
"ML: gradient boosting, no maintenance counters",0.51,"[0.48, 0.54]",34%,100%
"ML: gradient boosting, all features",0.30,"[0.26, 0.34]",18%,100%


<a id="appx-c4"></a>
### C.4 · Notebook 04: test and select
* **All ten ideas** were scored with lift, precision and coverage, with thresholds from normal operation only.
  Nine passed every gate; **C07 failed the causal check**. Why the others lose: section 2.
* **Kept:** **C09**, **C08** and **C10** (section 1 has their scores; appendix A.2 has the rules in full).
* **Smoothing:** a 6 h average works best; CUSUM is worst.
* **Choosing H and W:** one F1 grid of 100 windows (heatmap in [section 5.2](#s5-2)). F1 is flat
  for every window that contains the error code, and collapses once H reaches 36 h.
* **Every event type:** C09 ranks first for any failure and for each part separately.

In [14]:
# Saved by notebook 04: the shortlist for every event type (holdout machines), and the H x W grid's key numbers.
S = pd.DataFrame([dict(event=e, rank=k["kept"].index(c["cid"]) + 1, id=c["cid"], rule=rule_name(c),
                       F1=c["hold"]["f1"], precision=c["hold"]["precision"], coverage=c["hold"]["coverage"])
                  for e, k in N4["kinds"].items() for c in k["concepts"] if c["status"] == "kept"]).sort_values(["event", "rank"])
display(S.style.hide(axis="index").format({"F1": "{:.2f}", "precision": "{:.0%}", "coverage": "{:.0%}"}))
G = pd.DataFrame(N4["hw_grid"]).set_index("H")
G.columns = G.columns.astype(int)
print(f"H x W grid (dev F1): chosen H=12, W=36 -> {G.loc[12, 36]:.3f} | H=0, W=48 -> {G.loc[0, 48]:.3f} | "
      f"best anywhere -> {G.values.max():.3f} | any window with H >= 36: at most {G.loc[[36, 48, 72]].values.max():.2f}")

event,rank,id,rule,F1,precision,coverage
any,1,C09,Matched signature,0.94,90%,99%
any,2,C08,Drift + any error,0.74,63%,91%
any,3,C10,Drift on an aging part,0.56,41%,89%
comp1,1,C09,Matched signature,0.93,89%,97%
comp1,2,C10,Drift on an aging part,0.57,41%,93%
comp1,3,C01,Voltage creeps up,0.41,27%,92%
comp2,1,C09,Matched signature,0.94,90%,99%
comp2,2,C08,Drift + any error,0.78,69%,89%
comp2,3,C06,Any error code,0.23,62%,14%
comp3,1,C09,Matched signature,0.93,87%,100%


H x W grid (dev F1): chosen H=12, W=36 -> 0.936 | H=0, W=48 -> 0.945 | best anywhere -> 0.948 | any window with H >= 36: at most 0.24


**The window grid also fixed C09's settings.** In each cell of grid 1 ([section 5.2](#s5-2)), C09 is *re-tuned*: all 40 of its variants are tried on the dev machines and the one with the best dev F1 is kept. The four settings it chooses between:

| Setting | Options tried | Chosen (red cell) |
|---|---|---|
| Smoothing | 6 h or 24 h average | **6 h average** |
| Usual level | per-machine percentile, or robust (median + spreads) | **robust** |
| Cut-off | 1.64, 2.33, 2.58, 3.09 or 3.72 spreads (P95 to P99.99) | **3.09 spreads** (P99.9) |
| Error look-back | 24 h or 48 h | **24 h** |

So picking the red cell fixes everything at once: **the window (H = 12, W = 36) and the rule's four settings**, which together are exactly the C09 written out in appendix A.2 (dev F1 0.936 in both places). In other cells a different variant may win (for example 3.72 spreads, or a 48 h look-back); that is what lets grid 1 show each window at its best. Nothing is chosen after this step. Grid 2 only checks the result: it keeps these settings **fixed** and scores the rule on the holdout machines at every window.

### C.5 · Notebook 05: audit (trying to break the results)
Notebook 05 tries to break the shortlist with the classic traps. Each passes:
* **Luck from trying many ideas.** 300 rules built from pure random noise, tuned exactly like the real
  ones, score about **0.08** at best, against C09's 0.94 (chart below). Picking the best of many tries
  manufactures almost no skill.
* **Leakage through the failure hour.** Scoring the failure hour itself would turn "a repair was logged" into a
  fake perfect rule; the failure hour is never scored.
* **Counters that leak the future.** "Hours since maintenance" looks excellent until it may only use past data,
  then collapses; the gate rejects it (C07).
* **Time drift.** Rules set up on January–June alone still work on July–December (table below).
* **Different machines.** C09 works on every machine model and age band (precision 0.87–0.93, coverage 0.97–0.99).

In [15]:
# The luck check (notebook 05): 300 rules built from pure random noise, tuned exactly like the real rules.
L = N5["audits"]["lucky"]
c09 = next(c for c in K["concepts"] if c["cid"] == "C09")["dev"]["f1"]
fig = go.Figure(go.Histogram(x=L["dev_f1"], xbins=dict(start=0, end=1, size=0.01), marker_color=GREY,
                             name="rules built from random noise"))
fig.add_vline(x=c09, line_color=BLUE, line_width=3)
fig.add_annotation(x=c09, y=1, yref="paper", text=f"C09: {c09:.2f}", showarrow=False, xanchor="right", xshift=-6,
                   font=dict(color=BLUE, size=13))
fig.add_annotation(x=L["best_dev"], y=1, yref="paper", text=f"best of 300 noise rules: {L['best_dev']:.2f}",
                   showarrow=False, xanchor="left", xshift=8, font=dict(size=12))
look(fig, "300 rules built from random noise, tuned the same way, against C09 (dev F1)", 320)
fig.update_xaxes(title="dev F1", range=[0, 1]); fig.update_yaxes(title="number of noise rules")

In [16]:
# Saved by notebook 05: the luck check and the time-drift check, in plain terms.
L, T = N5["audits"]["lucky"], {t["cid"]: t for t in N5["audits"]["temporal"]}
pd.DataFrame([
    dict(check=f"Best of {len(L['dev_f1'])} rules built from random noise, picked on dev machines",
         result=f"F1 {L['best_dev']:.2f} on dev, {L['best_hold']:.2f} on holdout",
         meaning="trying many rules finds almost nothing by luck"),
    *[dict(check=f"{cid}: set up on Jan–Jun only, scored on Jul–Dec",
           result=f"F1 {T[cid]['first_half_thresholds']['f1']:.2f} (vs {T[cid]['full_year_thresholds']['f1']:.2f} set up on the full year)",
           meaning="still works on months it has not seen")
      for cid in ("C09", "C08", "C10")],
]).set_index("check")

,result,meaning
check,,
"Best of 300 rules built from random noise, picked on dev machines","F1 0.08 on dev, 0.08 on holdout",trying many rules finds almost nothing by luck
"C09: set up on Jan–Jun only, scored on Jul–Dec",F1 0.93 (vs 0.93 set up on the full year),still works on months it has not seen
"C08: set up on Jan–Jun only, scored on Jul–Dec",F1 0.74 (vs 0.73 set up on the full year),still works on months it has not seen
"C10: set up on Jan–Jun only, scored on Jul–Dec",F1 0.57 (vs 0.55 set up on the full year),still works on months it has not seen
